In [1]:
def prepare_or_load_bess_scm_data(
    data_path="data/Energy_dataset_bessScaled.csv.xz",
    cache_path="prepared_scm_data/bess_scm_data_minimal.pkl.xz",
    dataset_key="bess",
    date_column="date",
    netconsumption_prefix="netconsumption",
    treated_unit_idx=0,
    buildings_range=range(300),
    train_split=0.60,
    val_split=0.10,
    force_rebuild=False,
    store_float32=True,
    verbose=True,
):
    """
    Prepare or load minimal scm_data for the BESS SCM experiments.

    The returned object has the structure expected by
    run_scm_experiments_with_diagnostics and
    run_scm_lag_and_control_size_experiment_grid.

    Only the required objects are stored:
        Z0, Z1, Y0, Y1, meta

    Nothing else is kept in scm_data, so df, wide, X0, and X1 are not stored.
    """

    from pathlib import Path
    import hashlib
    import json
    import lzma
    import pickle

    import numpy as np
    import pandas as pd

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    def _json_safe(value):
        if isinstance(value, range):
            return list(value)
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, (list, tuple)):
            return list(value)
        return value

    preparation_settings = {
        "data_path": str(data_path),
        "dataset_key": str(dataset_key),
        "date_column": str(date_column),
        "netconsumption_prefix": str(netconsumption_prefix),
        "treated_unit_idx": int(treated_unit_idx),
        "buildings_range": _json_safe(buildings_range),
        "train_split": float(train_split),
        "val_split": float(val_split),
        "store_float32": bool(store_float32),
    }

    preparation_signature = hashlib.md5(
        json.dumps(preparation_settings, sort_keys=True).encode("utf-8")
    ).hexdigest()

    def _print_shapes(scm_data, source):
        out = scm_data[dataset_key]
        meta = out["meta"]

        print(f"{dataset_key} loaded from {source}")
        print(
            f"  treated={meta['treated_unit']} | "
            f"controls={len(meta['controls'])} | "
            f"pre={out['Z0'].shape[0]} | post={out['Y0'].shape[0]} | "
            f"split={meta['split_label']}"
        )
        print(
            f"  Z0{out['Z0'].shape} {out['Z0'].dtype} | "
            f"Z1{out['Z1'].shape} {out['Z1'].dtype} | "
            f"Y0{out['Y0'].shape} {out['Y0'].dtype} | "
            f"Y1{out['Y1'].shape} {out['Y1'].dtype}"
        )

    if cache_path.exists() and not force_rebuild:
        with lzma.open(cache_path, "rb") as file:
            scm_data = pickle.load(file)

        cached_signature = (
            scm_data
            .get(dataset_key, {})
            .get("meta", {})
            .get("preparation_signature", None)
        )

        if cached_signature == preparation_signature:
            if verbose:
                _print_shapes(scm_data, source=cache_path)
            return scm_data

        if verbose:
            print("Cached scm_data exists, but the preparation settings changed. Rebuilding.")

    data_path = Path(data_path)

    header = pd.read_csv(data_path, nrows=0)
    all_columns = list(header.columns)

    netconsumption_cols = [
        col for col in all_columns
        if str(col).startswith(netconsumption_prefix)
    ]

    if len(netconsumption_cols) < 2:
        raise ValueError(
            f"Need at least two columns starting with {netconsumption_prefix!r}. "
            f"Found {len(netconsumption_cols)}."
        )

    def _netconsumption_suffix(column_name):
        return int(str(column_name).replace(f"{netconsumption_prefix}_", ""))

    netconsumption_cols = sorted(netconsumption_cols, key=_netconsumption_suffix)

    if buildings_range is not None:
        allowed_suffixes = {int(i) + 1 for i in buildings_range}

        netconsumption_cols = [
            col for col in netconsumption_cols
            if _netconsumption_suffix(col) in allowed_suffixes
        ]

    if len(netconsumption_cols) < 2:
        raise ValueError("Too few net-consumption columns after applying buildings_range.")

    if treated_unit_idx < 0 or treated_unit_idx >= len(netconsumption_cols):
        raise ValueError(
            f"treated_unit_idx={treated_unit_idx} is invalid for "
            f"{len(netconsumption_cols)} selected net-consumption columns."
        )

    treated_unit = netconsumption_cols[int(treated_unit_idx)]

    controls = [
        col for col in netconsumption_cols
        if col != treated_unit
    ]

    usecols = [date_column, treated_unit] + controls

    df = pd.read_csv(
        data_path,
        usecols=usecols,
        parse_dates=[date_column],
    )

    df = df.sort_values(date_column).reset_index(drop=True)

    if df[usecols].isna().any().any():
        missing_columns = df[usecols].columns[df[usecols].isna().any()].tolist()
        raise ValueError(f"Missing values found in required columns: {missing_columns}")

    pre_fraction = float(train_split) + float(val_split)

    if not (0.0 < train_split < 1.0):
        raise ValueError("train_split must be between 0 and 1.")

    if not (0.0 <= val_split < 1.0):
        raise ValueError("val_split must be between 0 and 1.")

    if not (0.0 < pre_fraction < 1.0):
        raise ValueError("train_split + val_split must be between 0 and 1.")

    pre_split_idx = int(len(df) * pre_fraction)

    if pre_split_idx < 2:
        raise ValueError("Too few pre-treatment observations.")

    if len(df) - pre_split_idx < 2:
        raise ValueError("Too few post-treatment observations.")

    dtype = np.float32 if store_float32 else np.float64

    Z1 = df.loc[: pre_split_idx - 1, treated_unit].to_numpy(dtype=dtype).reshape(-1, 1)
    Z0 = df.loc[: pre_split_idx - 1, controls].to_numpy(dtype=dtype)

    Y1 = df.loc[pre_split_idx:, treated_unit].to_numpy(dtype=dtype).reshape(-1, 1)
    Y0 = df.loc[pre_split_idx:, controls].to_numpy(dtype=dtype)

    time_index_pre = pd.Index(df.loc[: pre_split_idx - 1, date_column])
    time_index_post = pd.Index(df.loc[pre_split_idx:, date_column])

    scm_data = {
        dataset_key: {
            "Z0": Z0,
            "Z1": Z1,
            "Y0": Y0,
            "Y1": Y1,
            "meta": {
                "dataset": dataset_key,
                "kind": "bess_panel_features",
                "treated_unit": treated_unit,
                "controls": controls,
                "time_variable": date_column,
                "unit_variable": "wide_columns",
                "outcome_variable": treated_unit,
                "intervention_time": int(pre_split_idx),
                "plot_intervention_time": time_index_post[0],
                "split_label": str(time_index_post[0]),
                "time_index_pre": time_index_pre,
                "time_index_post": time_index_post,
                "train_split": float(train_split),
                "val_split": float(val_split),
                "pre_fraction": float(pre_fraction),
                "treated_unit_idx": int(treated_unit_idx),
                "buildings_range": list(buildings_range) if buildings_range is not None else None,
                "preparation_settings": preparation_settings,
                "preparation_signature": preparation_signature,
            },
        }
    }

    with lzma.open(cache_path, "wb") as file:
        pickle.dump(scm_data, file, protocol=pickle.HIGHEST_PROTOCOL)

    if verbose:
        _print_shapes(scm_data, source=f"newly prepared and saved to {cache_path}")

    return scm_data


# ============================================================
# First execution
# ============================================================

scm_data = prepare_or_load_bess_scm_data(
    data_path="data/Energy_dataset_bessScaled.csv.xz",
    cache_path="prepared_scm_data/bess_scm_data_minimal.pkl.xz",
    dataset_key="bess",
    treated_unit_idx=0,
    buildings_range=range(300),
    train_split=0.60,
    val_split=0.10,
    force_rebuild=False,
    store_float32=True,
    verbose=True,
)

bess loaded from newly prepared and saved to prepared_scm_data\bess_scm_data_minimal.pkl.xz
  treated=netconsumption_1 | controls=299 | pre=36825 | post=15783 | split=2012-08-06 05:00:00
  Z0(36825, 299) float32 | Z1(36825, 1) float32 | Y0(15783, 299) float32 | Y1(15783, 1) float32


In [2]:
import gc
import ctypes
import matplotlib.pyplot as plt

import psutil

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / 1024**3:.2f} GB")
print(f"Used RAM:      {memory.used / 1024**3:.2f} GB")
print(f"RAM usage:     {memory.percent:.1f} %")

# Close all matplotlib figures.
plt.close("all")

# Clear TensorFlow and Keras state if TensorFlow was imported.
try:
    import tensorflow as tf
    tf.keras.backend.clear_session()
except Exception:
    pass

# Remove common large result objects from previous runs.
for name in [
    "all_results",
    "grid_results",
    "grid_metrics_df",
    "grid_diagnostic_summary_df",
    "experiment_plan_df",
    "grid_runtime_log_df",
    "diagnostic_summary_df",
    "all_selected_metrics_df",
]:
    if name in globals():
        del globals()[name]

# Clear IPython output cache. This is important in notebooks.
try:
    ip = get_ipython()
    ip.run_line_magic("reset", "-f out")
except Exception:
    pass

# Run Python garbage collection.
gc.collect()

# On Linux/Raspberry Pi, ask libc to return freed heap memory to the OS.
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

gc.collect()

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / 1024**3:.2f} GB")
print(f"Used RAM:      {memory.used / 1024**3:.2f} GB")
print(f"RAM usage:     {memory.percent:.1f} %")

Available RAM: 47.52 GB
Used RAM:      16.39 GB
RAM usage:     25.6 %


Flushing output cache (0 entries)
Available RAM: 47.27 GB
Used RAM:      16.64 GB
RAM usage:     26.0 %


In [3]:
def run_scm_experiments_with_diagnostics(
    scm_data,
    datasets=None,
    lambda_grid=(0.0001, 0.001, 0.01, 0.1, 1.0),
    nn_lambda_grid=None,
    sum_to_one_grid=(True, False),
    positive_weights_constraint=(True, False),
    enable_linear_scm=True,
    enable_nn_scm=False,
    use_lagged_treated=False,
    lagged_treated_steps=24,
    use_topk_donor_lags=False,
    donor_lag_past_steps=24,
    donor_lag_top_k=1,
    control_group_size=None,
    random_state=42,
    pre_validation_fraction=0.2,
    pre_test_fraction=0.1,
    use_cross_validation_if_small=True,
    min_validation_rows=3,
    min_test_rows=3,
    cv_folds=5,
    standardize_linear_features=False,
    standardize_nn_features=True,
    nn_hidden_units=(16, 16),
    nn_learning_rate=1e-4,
    nn_epochs=2000,
    nn_batch_size=32,
    nn_early_stopping_patience=80,
    enable_placebo_tests=False,
    max_placebos=None,
    enable_jackknife_tests=False,
    max_jackknife_donors=None,
    pre_gap_trend_seasonal_period=None,
    save_results=True,
    save_dir="results_scm_diagnostics",
    plot_results=True,
    plot_width=13,
    plot_height=4.2,
    plot_robustness_results=True,
    trend_smoothing_window_by_dataset=None,
    smooth_plot_datasets=("bess", "london"),
    compact_summary=True,
    summary_round_digits=6,
):
    """
    Run linear and optional neural network Synthetic Control Method experiments.

    Vocabulary used in the output
    -----------------------------
    training:
        The first part of the pre-treatment period used for hyperparameter selection.

    validation:
        The pre-treatment validation part used for hyperparameter selection.
        If the validation or test split is too small and cross validation is enabled,
        this becomes blocked K-fold validation on the pre-test training pool.

    test:
        The last pre-treatment part used only after hyperparameter selection.

    pre:
        The full effective pre-treatment period used for the final refit.

    post:
        The post-treatment period.

    Important convention
    --------------------
    gap = observed treated outcome - synthetic counterfactual

    A positive post gap means the observed treated outcome is above its estimated
    counterfactual.

    The function intentionally does not catch broad errors.
    If something is inconsistent, Python raises the original error.
    """

    from pathlib import Path
    import hashlib
    import time

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from scipy.optimize import nnls

    assert enable_linear_scm or enable_nn_scm, "At least one model family must be enabled."

    keras = None
    if enable_nn_scm:
        from tensorflow import keras as keras_module
        keras = keras_module

    save_dir = Path(save_dir)

    if datasets is None:
        datasets = list(scm_data.keys())

    if isinstance(datasets, str):
        datasets = [datasets]

    if nn_lambda_grid is None:
        nn_lambda_grid = lambda_grid

    lambda_grid = sorted({float(value) for value in lambda_grid})
    nn_lambda_grid = sorted({float(value) for value in nn_lambda_grid})

    if save_results:
        save_dir.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------------------
    # Small helpers
    # ---------------------------------------------------------------------
    def _as_1d(values):
        return np.asarray(values, dtype=float).reshape(-1)

    def _elapsed_seconds(start_time):
        return float(time.perf_counter() - start_time)

    def _safe_scale(values):
        values = _as_1d(values)
        scale = float(np.std(values, ddof=0))
        return scale if scale > 1e-12 else 1.0

    def _safe_divide(numerator, denominator):
        denominator = float(denominator)
        if abs(denominator) <= 1e-12:
            return np.nan
        return float(numerator / denominator)

    def _coefficient_of_variation(values):
        values = _as_1d(values)
        values = values[np.isfinite(values)]

        if len(values) == 0:
            return np.nan

        mean_value = float(np.mean(values))
        std_value = float(np.std(values, ddof=0))

        return _safe_divide(std_value, abs(mean_value))

    def _normalise_bool_grid(value):
        if isinstance(value, (bool, np.bool_)):
            return (bool(value),)

        if isinstance(value, (list, tuple, set)):
            return tuple(dict.fromkeys(bool(v) for v in value))

        return (bool(value),)

    def _stable_int_from_text(text):
        digest = hashlib.md5(str(text).encode("utf-8")).hexdigest()
        return int(digest[:8], 16)

    def _safe_abs_corr(a, b):
        a = _as_1d(a)
        b = _as_1d(b)

        mask = np.isfinite(a) & np.isfinite(b)
        a = a[mask]
        b = b[mask]

        if len(a) < 2:
            return 0.0

        a_std = float(np.std(a, ddof=0))
        b_std = float(np.std(b, ddof=0))

        if a_std < 1e-12 or b_std < 1e-12:
            return 0.0

        a_centered = a - float(np.mean(a))
        b_centered = b - float(np.mean(b))

        return abs(float(np.mean(a_centered * b_centered) / (a_std * b_std)))

    def _dataset_option(option, dataset_key, default=None):
        if isinstance(option, dict):
            if dataset_key in option:
                return option[dataset_key]
            if str(dataset_key).lower() in option:
                return option[str(dataset_key).lower()]
            return default

        if option is None:
            return default

        return option

    def _fmt(value, digits=4):
        if value is None:
            return "NA"

        if isinstance(value, str):
            return value

        if pd.isna(value):
            return "NA"

        return f"{float(value):.{digits}f}"

    sum_to_one_grid_local = _normalise_bool_grid(sum_to_one_grid)
    positive_weights_grid = _normalise_bool_grid(positive_weights_constraint)

    # ---------------------------------------------------------------------
    # Metrics
    # ---------------------------------------------------------------------
    def _error_metrics(y_true, y_pred, prefix, scale_value):
        """
        Compute real-unit and standard-deviation-scaled metrics.

        gap = observed - synthetic
        """
        y_true = _as_1d(y_true)
        y_pred = _as_1d(y_pred)

        gaps = y_true - y_pred
        abs_gaps = np.abs(gaps)
        squared_gaps = gaps ** 2

        scaled_gaps = gaps / float(scale_value)
        abs_scaled_gaps = np.abs(scaled_gaps)
        squared_scaled_gaps = scaled_gaps ** 2

        mean_gap = float(np.mean(gaps))
        scaled_mean_gap = float(np.mean(scaled_gaps))

        if abs(mean_gap) <= 1e-12:
            same_sign_share = np.nan
            effect_direction = "zero"
        else:
            effect_sign = np.sign(mean_gap)
            same_sign_share = float(np.mean(np.sign(gaps) == effect_sign))
            effect_direction = "positive" if effect_sign > 0 else "negative"

        cumulative_gap = float(np.sum(gaps))
        scaled_cumulative_gap = float(np.sum(scaled_gaps))

        metrics = {
            f"{prefix}_mse": float(np.mean(squared_gaps)),
            f"{prefix}_mae": float(np.mean(abs_gaps)),
            f"{prefix}_rmse": float(np.sqrt(np.mean(squared_gaps))),
            f"{prefix}_sum_squared_error": float(np.sum(squared_gaps)),
            f"{prefix}_sum_absolute_error": float(np.sum(abs_gaps)),
            f"{prefix}_root_sum_squared_error": float(np.sqrt(np.sum(squared_gaps))),
            f"{prefix}_mean_gap": mean_gap,
            f"{prefix}_abs_mean_gap": abs(mean_gap),
            f"{prefix}_cumulative_gap": cumulative_gap,
            f"{prefix}_abs_cumulative_gap": abs(cumulative_gap),
            f"{prefix}_same_sign_gap_share": same_sign_share,
            f"{prefix}_effect_direction": effect_direction,

            f"{prefix}_scaled_mse": float(np.mean(squared_scaled_gaps)),
            f"{prefix}_scaled_mae": float(np.mean(abs_scaled_gaps)),
            f"{prefix}_scaled_rmse": float(np.sqrt(np.mean(squared_scaled_gaps))),
            f"{prefix}_scaled_sum_squared_error": float(np.sum(squared_scaled_gaps)),
            f"{prefix}_scaled_sum_absolute_error": float(np.sum(abs_scaled_gaps)),
            f"{prefix}_scaled_root_sum_squared_error": float(np.sqrt(np.sum(squared_scaled_gaps))),
            f"{prefix}_scaled_mean_gap": scaled_mean_gap,
            f"{prefix}_abs_scaled_mean_gap": abs(scaled_mean_gap),
            f"{prefix}_scaled_cumulative_gap": scaled_cumulative_gap,
            f"{prefix}_abs_scaled_cumulative_gap": abs(scaled_cumulative_gap),
        }

        metrics[f"{prefix}_scaled_mse_to_mae_ratio"] = _safe_divide(
            metrics[f"{prefix}_scaled_mse"],
            metrics[f"{prefix}_scaled_mae"],
        )

        metrics[f"{prefix}_scaled_rmse_to_mae_ratio"] = _safe_divide(
            metrics[f"{prefix}_scaled_rmse"],
            metrics[f"{prefix}_scaled_mae"],
        )

        return metrics

    def _add_test_train_ratios(metrics):
        metrics["test_train_mse_ratio"] = _safe_divide(
            metrics["test_mse"],
            metrics["training_mse"],
        )

        metrics["test_train_mae_ratio"] = _safe_divide(
            metrics["test_mae"],
            metrics["training_mae"],
        )

        metrics["test_train_scaled_mse_ratio"] = _safe_divide(
            metrics["test_scaled_mse"],
            metrics["training_scaled_mse"],
        )

        metrics["test_train_scaled_mae_ratio"] = _safe_divide(
            metrics["test_scaled_mae"],
            metrics["training_scaled_mae"],
        )

    def _add_post_pre_ratios(metrics, pre_prefix, post_prefix, ratio_prefix):
        for name in [
            "mse",
            "mae",
            "rmse",
            "sum_squared_error",
            "sum_absolute_error",
            "root_sum_squared_error",
            "scaled_mse",
            "scaled_mae",
            "scaled_rmse",
            "scaled_sum_squared_error",
            "scaled_sum_absolute_error",
            "scaled_root_sum_squared_error",
        ]:
            metrics[f"{ratio_prefix}_{name}_ratio"] = _safe_divide(
                metrics[f"{post_prefix}_{name}"],
                metrics[f"{pre_prefix}_{name}"],
            )

    def _infer_seasonal_period(time_index_pre, dataset_key):
        requested_period = _dataset_option(
            pre_gap_trend_seasonal_period,
            dataset_key,
            default=None,
        )

        if requested_period is not None:
            requested_period = int(requested_period)
            return requested_period if requested_period > 1 else None

        time_index_pre = pd.Index(time_index_pre)

        if not pd.api.types.is_datetime64_any_dtype(time_index_pre):
            return None

        if len(time_index_pre) < 3:
            return None

        deltas = pd.Series(time_index_pre).diff().dropna()
        median_delta_seconds = float(deltas.median().total_seconds())

        if median_delta_seconds <= 0:
            return None

        seconds_per_day = 24 * 60 * 60

        if median_delta_seconds < seconds_per_day:
            candidate = int(round(seconds_per_day / median_delta_seconds))
            return candidate if candidate > 1 else None

        if 0.75 * seconds_per_day <= median_delta_seconds <= 1.25 * seconds_per_day:
            return 7

        return None

    def _add_pre_gap_trend(metrics, pre_gaps, scale_value, seasonal_period):
        """
        Estimate drift in the full pre-treatment gap.
        """
        pre_gaps = _as_1d(pre_gaps)
        scaled_gaps = pre_gaps / float(scale_value)

        if seasonal_period is not None and int(seasonal_period) > 1:
            seasonal_period = int(seasonal_period)
            seasonal_position = np.arange(len(scaled_gaps)) % seasonal_period
            seasonal_mean = (
                pd.DataFrame(
                    {
                        "gap": scaled_gaps,
                        "seasonal_position": seasonal_position,
                    }
                )
                .groupby("seasonal_position")["gap"]
                .transform("mean")
                .to_numpy(dtype=float)
            )
            adjusted_gaps = scaled_gaps - seasonal_mean
        else:
            seasonal_period = np.nan
            adjusted_gaps = scaled_gaps.copy()

        if len(adjusted_gaps) < 2:
            slope_per_step = np.nan
            total_trend = np.nan
        else:
            x = np.arange(len(adjusted_gaps), dtype=float)
            x_centered = x - float(np.mean(x))
            slope_per_step = float(np.polyfit(x_centered, adjusted_gaps, deg=1)[0])
            total_trend = float(slope_per_step * (len(adjusted_gaps) - 1))

        metrics["pre_gap_trend_seasonal_period"] = seasonal_period
        metrics["pre_gap_trend_scaled_per_step"] = slope_per_step
        metrics["pre_gap_trend_scaled_total"] = total_trend

    # ---------------------------------------------------------------------
    # Feature construction
    # ---------------------------------------------------------------------
    def _classify_feature(feature_name):
        feature_name = str(feature_name)

        if "__treated_lag_" in feature_name:
            return "treated_lag"

        if "__donor_lag_" in feature_name:
            return "donor_lag"

        return "current_donor"

    def _extract_donor_name_from_feature(feature_name):
        feature_name = str(feature_name)

        if "__donor_lag_" in feature_name:
            return feature_name.split("__donor_lag_", 1)[0]

        if "__treated_lag_" in feature_name:
            return None

        return feature_name


    def _parse_donor_lag_feature(feature_name):
        feature_name = str(feature_name)
        donor_name, lag_text = feature_name.rsplit("__donor_lag_", 1)
        return donor_name, int(lag_text)


    def _parse_treated_lag_feature(feature_name):
        feature_name = str(feature_name)
        return int(feature_name.rsplit("__treated_lag_", 1)[1])


    def _make_lag_frames(y_all, X_all_current):
        """
        Low-memory lag construction.

        This function no longer materializes full lag matrices. It only creates the
        feature names and donor-lag lookup map. Actual lagged values are generated
        lazily in _collect_raw_features and _prepare_design.
        """
        X_treated_lags = pd.DataFrame(index=X_all_current.index)
        X_donor_lags = pd.DataFrame(index=X_all_current.index)

        treated_lag_names = []

        if use_lagged_treated:
            treated_lag_names = [
                f"treated__treated_lag_{lag}"
                for lag in range(1, int(lagged_treated_steps) + 1)
            ]

        donor_lag_map = {donor: [] for donor in X_all_current.columns}

        if use_topk_donor_lags:
            for donor in X_all_current.columns:
                donor_lag_map[donor] = [
                    f"{donor}__donor_lag_{lag}"
                    for lag in range(1, int(donor_lag_past_steps) + 1)
                ]

        return X_donor_lags, X_treated_lags, donor_lag_map, treated_lag_names


    def _collect_raw_features(
        positions,
        feature_names,
        X_all_current,
        X_all_donor_lags,
        X_all_treated_lags,
    ):
        """
        Collect raw feature values.

        Donor lags and treated lags are generated lazily instead of being stored in
        large DataFrames. This is the main memory-saving change.
        """
        data = {}

        for feature_name in feature_names:
            feature_name = str(feature_name)

            if feature_name in X_all_current.columns:
                values = X_all_current.loc[positions, feature_name].to_numpy(dtype=float)

            elif "__donor_lag_" in feature_name:
                donor_name, lag = _parse_donor_lag_feature(feature_name)
                values = (
                    X_all_current[donor_name]
                    .shift(lag)
                    .loc[positions]
                    .to_numpy(dtype=float)
                )

            elif "__treated_lag_" in feature_name:
                lag = _parse_treated_lag_feature(feature_name)
                values = (
                    y_all
                    .shift(lag)
                    .loc[positions]
                    .to_numpy(dtype=float)
                )

            else:
                raise ValueError(f"Unknown feature name: {feature_name}")

            if not np.isfinite(values).all():
                raise ValueError(
                    f"Feature {feature_name!r} contains non-finite values. "
                    "Check whether the initial lagged rows were dropped correctly."
                )

            data[feature_name] = values

        return pd.DataFrame(data, index=positions)


    def _prepare_design(
        train_positions,
        predict_positions,
        y_all,
        X_all_current,
        X_all_donor_lags,
        X_all_treated_lags,
        current_feature_names,
        donor_lag_map,
        treated_lag_names,
        standardize_model_data,
    ):
        y_train_raw = y_all.loc[train_positions].to_numpy(dtype=float)
        y_predict_raw = y_all.loc[predict_positions].to_numpy(dtype=float)

        selected_donor_lag_names = []

        if use_topk_donor_lags:
            y_train_values = y_all.loc[train_positions].to_numpy(dtype=float)

            for donor_name in current_feature_names:
                lag_scores = []

                for lag_name in donor_lag_map[donor_name]:
                    _, lag = _parse_donor_lag_feature(lag_name)

                    lag_values = (
                        X_all_current[donor_name]
                        .shift(lag)
                        .loc[train_positions]
                        .to_numpy(dtype=float)
                    )

                    lag_scores.append((lag_name, _safe_abs_corr(y_train_values, lag_values)))

                lag_scores.sort(key=lambda item: item[1], reverse=True)

                selected_donor_lag_names.extend(
                    [name for name, _ in lag_scores[: int(donor_lag_top_k)]]
                )

        feature_names = []
        feature_names.extend(list(current_feature_names))
        feature_names.extend(selected_donor_lag_names)

        if use_lagged_treated:
            feature_names.extend(treated_lag_names)

        feature_names = list(dict.fromkeys(feature_names))

        X_train_raw = _collect_raw_features(
            positions=train_positions,
            feature_names=feature_names,
            X_all_current=X_all_current,
            X_all_donor_lags=X_all_donor_lags,
            X_all_treated_lags=X_all_treated_lags,
        )

        X_predict_raw = _collect_raw_features(
            positions=predict_positions,
            feature_names=feature_names,
            X_all_current=X_all_current,
            X_all_donor_lags=X_all_donor_lags,
            X_all_treated_lags=X_all_treated_lags,
        )

        feature_std = X_train_raw.std(axis=0, ddof=0)
        nonconstant_feature_names = feature_std[feature_std > 1e-12].index.tolist()

        X_train_raw = X_train_raw[nonconstant_feature_names]
        X_predict_raw = X_predict_raw[nonconstant_feature_names]
        feature_names = nonconstant_feature_names

        if len(feature_names) == 0:
            raise ValueError("No usable features left after removing constant features.")

        if standardize_model_data:
            X_mean = X_train_raw.mean(axis=0)
            X_std = X_train_raw.std(axis=0, ddof=0)
            X_std[X_std < 1e-12] = 1.0

            y_mean = float(np.mean(y_train_raw))
            y_std = float(np.std(y_train_raw, ddof=0))
            y_std = y_std if y_std > 1e-12 else 1.0

            X_train_model = ((X_train_raw - X_mean) / X_std).to_numpy(dtype=float)
            X_predict_model = ((X_predict_raw - X_mean) / X_std).to_numpy(dtype=float)

            y_train_model = (y_train_raw - y_mean) / y_std
            y_predict_model = (y_predict_raw - y_mean) / y_std

        else:
            X_mean = pd.Series(0.0, index=feature_names)
            X_std = pd.Series(1.0, index=feature_names)
            y_mean = 0.0
            y_std = 1.0

            X_train_model = X_train_raw.to_numpy(dtype=float)
            X_predict_model = X_predict_raw.to_numpy(dtype=float)

            y_train_model = y_train_raw.copy()
            y_predict_model = y_predict_raw.copy()

        return {
            "feature_names": feature_names,
            "X_mean": X_mean,
            "X_std": X_std,
            "y_mean": y_mean,
            "y_std": y_std,
            "standardize": bool(standardize_model_data),
            "X_train_raw": X_train_raw,
            "X_predict_raw": X_predict_raw,
            "y_train_raw": y_train_raw,
            "y_predict_raw": y_predict_raw,
            "X_train_model": X_train_model,
            "X_predict_model": X_predict_model,
            "y_train_model": y_train_model,
            "y_predict_model": y_predict_model,
        }

    # ---------------------------------------------------------------------
    # Model fitting
    # ---------------------------------------------------------------------
    def _fit_linear_scm(
        prepared,
        lambda_value,
        sum_to_one_constraint,
        positive_weights_constraint_value,
    ):
        X_train = prepared["X_train_model"]
        y_train = prepared["y_train_model"]

        n_samples, n_features = X_train.shape

        residual_scale = float(np.std(y_train, ddof=0))
        residual_scale = residual_scale if residual_scale > 1e-12 else 1.0

        X_scaled = X_train / residual_scale
        y_scaled = y_train / residual_scale

        lambda_value = float(lambda_value)
        sum_to_one_constraint = bool(sum_to_one_constraint)
        positive_weights_constraint_value = bool(positive_weights_constraint_value)

        Q = (X_scaled.T @ X_scaled) / float(n_samples)
        c = (X_scaled.T @ y_scaled) / float(n_samples)

        if lambda_value > 0.0:
            Q = Q + lambda_value * np.eye(n_features)

        numerical_scale = max(
            1.0,
            float(np.max(np.abs(Q))),
            float(np.max(np.abs(c))),
        )

        Q_solver = Q / numerical_scale
        c_solver = c / numerical_scale

        def _project_to_simplex(values):
            values = np.asarray(values, dtype=float)

            sorted_values = np.sort(values)[::-1]
            cumulative = np.cumsum(sorted_values) - 1.0
            indices = np.arange(1, len(values) + 1)

            active = sorted_values - cumulative / indices > 0.0
            rho = indices[active][-1]
            theta = cumulative[active][-1] / float(rho)

            return np.maximum(values - theta, 0.0)

        def _solve_unconstrained():
            return np.linalg.pinv(Q_solver) @ c_solver

        def _solve_sum_to_one():
            ones = np.ones(n_features)

            kkt_matrix = np.block(
                [
                    [Q_solver, ones.reshape(-1, 1)],
                    [ones.reshape(1, -1), np.zeros((1, 1))],
                ]
            )

            kkt_rhs = np.concatenate([c_solver, np.array([1.0])])
            solution = np.linalg.pinv(kkt_matrix) @ kkt_rhs

            return solution[:n_features]

        def _solve_nonnegative():
            A_top = X_scaled / np.sqrt(float(n_samples))
            b_top = y_scaled / np.sqrt(float(n_samples))

            if lambda_value > 0.0:
                A_bottom = np.sqrt(lambda_value) * np.eye(n_features)
                b_bottom = np.zeros(n_features)

                A_augmented = np.vstack([A_top, A_bottom])
                b_augmented = np.concatenate([b_top, b_bottom])
            else:
                A_augmented = A_top
                b_augmented = b_top

            weights, _ = nnls(A_augmented, b_augmented)

            return weights

        def _solve_simplex():
            weights = np.full(n_features, 1.0 / float(n_features))

            largest_eigenvalue = float(np.linalg.eigvalsh(Q_solver).max())
            largest_eigenvalue = largest_eigenvalue if largest_eigenvalue > 1e-12 else 1.0

            step_size = 1.0 / (2.0 * largest_eigenvalue)
            max_iterations = 100_000
            tolerance = 1e-11

            for _ in range(max_iterations):
                gradient = 2.0 * (Q_solver @ weights - c_solver)
                next_weights = _project_to_simplex(weights - step_size * gradient)

                change = np.linalg.norm(next_weights - weights)
                scale = 1.0 + np.linalg.norm(weights)

                weights = next_weights

                if change <= tolerance * scale:
                    break

            return weights

        if positive_weights_constraint_value and sum_to_one_constraint:
            weights = _solve_simplex()

        elif positive_weights_constraint_value and not sum_to_one_constraint:
            weights = _solve_nonnegative()

        elif not positive_weights_constraint_value and sum_to_one_constraint:
            weights = _solve_sum_to_one()

        else:
            weights = _solve_unconstrained()

        if positive_weights_constraint_value:
            weights[np.abs(weights) < 1e-14] = 0.0

        if positive_weights_constraint_value and sum_to_one_constraint:
            weight_sum = float(np.sum(weights))
            if weight_sum <= 1e-12:
                weights = np.full(n_features, 1.0 / float(n_features))
            else:
                weights = weights / weight_sum

        residuals = y_scaled - X_scaled @ weights
        objective_value = float(
            np.mean(residuals ** 2)
            + lambda_value * np.sum(weights ** 2)
        )

        return {
            "model_type": "SCM",
            "lambda": float(lambda_value),
            "sum_to_one_constraint": bool(sum_to_one_constraint),
            "positive_weights_constraint": bool(positive_weights_constraint_value),
            "feature_names": prepared["feature_names"],
            "weights": weights.astype(float),
            "optimizer": {
                "solver": "pinv_nnls_or_projected_gradient",
                "objective": objective_value,
            },
            "X_mean": prepared["X_mean"],
            "X_std": prepared["X_std"],
            "y_mean": prepared["y_mean"],
            "y_std": prepared["y_std"],
            "standardize": prepared["standardize"],
        }

    def _build_nn_model(n_features, lambda_value):
        regularizer = keras.regularizers.l2(float(lambda_value)) if float(lambda_value) > 0 else None

        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(n_features,)))

        for hidden_units in nn_hidden_units:
            model.add(
                keras.layers.Dense(
                    int(hidden_units),
                    activation="relu",
                    kernel_regularizer=regularizer,
                )
            )

        model.add(keras.layers.Dense(1))

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=float(nn_learning_rate)),
            loss="mse",
        )

        return model

    def _fit_nn_scm(prepared, lambda_value, seed, fixed_epochs=None):
        keras.backend.clear_session()
        keras.utils.set_random_seed(int(seed))

        X_train = np.asarray(prepared["X_train_model"], dtype=np.float32)
        y_train = np.asarray(prepared["y_train_model"], dtype=np.float32)

        X_validation = np.asarray(prepared["X_predict_model"], dtype=np.float32)
        y_validation = np.asarray(prepared["y_predict_model"], dtype=np.float32)

        model = _build_nn_model(X_train.shape[1], lambda_value)

        if fixed_epochs is None:
            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=int(nn_early_stopping_patience),
                    restore_best_weights=True,
                    verbose=0,
                )
            ]

            history = model.fit(
                X_train,
                y_train,
                validation_data=(X_validation, y_validation),
                epochs=int(nn_epochs),
                batch_size=int(nn_batch_size),
                shuffle=False,
                verbose=0,
                callbacks=callbacks,
            )

            best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

        else:
            fixed_epochs = max(1, int(fixed_epochs))

            model.fit(
                X_train,
                y_train,
                epochs=fixed_epochs,
                batch_size=int(nn_batch_size),
                shuffle=False,
                verbose=0,
            )

            best_epoch = fixed_epochs

        return {
            "model_type": "NN-SCM",
            "lambda": float(lambda_value),
            "sum_to_one_constraint": np.nan,
            "positive_weights_constraint": np.nan,
            "feature_names": prepared["feature_names"],
            "model": model,
            "best_epoch": best_epoch,
            "X_mean": prepared["X_mean"],
            "X_std": prepared["X_std"],
            "y_mean": prepared["y_mean"],
            "y_std": prepared["y_std"],
            "standardize": prepared["standardize"],
        }

    def _fit_model_from_spec(model_name, prepared, spec, seed, fixed_epochs=None):
        if model_name == "SCM":
            return _fit_linear_scm(
                prepared=prepared,
                lambda_value=float(spec["lambda"]),
                sum_to_one_constraint=bool(spec["sum_to_one_constraint"]),
                positive_weights_constraint_value=bool(spec["positive_weights_constraint"]),
            )

        if model_name == "NN-SCM":
            epochs = fixed_epochs
            if epochs is None and "best_epoch" in spec and pd.notna(spec["best_epoch"]):
                epochs = int(spec["best_epoch"])

            return _fit_nn_scm(
                prepared=prepared,
                lambda_value=float(spec["lambda"]),
                seed=seed,
                fixed_epochs=epochs,
            )

        raise ValueError(f"Unknown model name: {model_name}")

    def _predict_with_fit(fit, X_raw):
        if fit["standardize"]:
            X_model = ((X_raw - fit["X_mean"]) / fit["X_std"]).to_numpy(dtype=float)
        else:
            X_model = X_raw.to_numpy(dtype=float)

        if fit["model_type"] == "SCM":
            prediction_model = X_model @ fit["weights"]

        elif fit["model_type"] == "NN-SCM":
            X_model = np.asarray(X_model, dtype=np.float32)
            prediction_model = fit["model"].predict(X_model, verbose=0).reshape(-1)

        else:
            raise ValueError(f"Unknown model type: {fit['model_type']}")

        if fit["standardize"]:
            return fit["y_mean"] + fit["y_std"] * prediction_model

        return prediction_model

    def _predict_one_row_with_fit(fit, row_raw):
        row_df = pd.DataFrame([row_raw], columns=fit["feature_names"])

        if fit["standardize"]:
            row_model = ((row_df - fit["X_mean"]) / fit["X_std"]).to_numpy(dtype=float)
        else:
            row_model = row_df.to_numpy(dtype=float)

        if fit["model_type"] == "SCM":
            prediction_model = float(row_model.reshape(-1) @ fit["weights"])

        elif fit["model_type"] == "NN-SCM":
            row_model = np.asarray(row_model, dtype=np.float32)
            prediction_model = float(fit["model"].predict(row_model, verbose=0).reshape(-1)[0])

        else:
            raise ValueError(f"Unknown model type: {fit['model_type']}")

        if fit["standardize"]:
            return float(fit["y_mean"] + fit["y_std"] * prediction_model)

        return prediction_model
    
    def _predict_post_period(
        fit,
        y_all,
        X_all_current,
        X_all_donor_lags,
        X_all_treated_lags,
        post_positions,
        n_pre_total,
    ):
        uses_treated_lags = any("__treated_lag_" in str(name) for name in fit["feature_names"])

        if not uses_treated_lags:
            X_post_raw = _collect_raw_features(
                positions=post_positions,
                feature_names=fit["feature_names"],
                X_all_current=X_all_current,
                X_all_donor_lags=X_all_donor_lags,
                X_all_treated_lags=X_all_treated_lags,
            )

            return _predict_with_fit(fit, X_post_raw)

        history = y_all.loc[: n_pre_total - 1].to_list()
        predictions = []

        for position in post_positions:
            row = []

            for feature_name in fit["feature_names"]:
                feature_name = str(feature_name)

                if "__treated_lag_" in feature_name:
                    lag = _parse_treated_lag_feature(feature_name)
                    row.append(float(history[-lag]))

                elif "__donor_lag_" in feature_name:
                    donor_name, lag = _parse_donor_lag_feature(feature_name)
                    row.append(float(X_all_current.loc[int(position) - lag, donor_name]))

                elif feature_name in X_all_current.columns:
                    row.append(float(X_all_current.loc[position, feature_name]))

                else:
                    raise ValueError(f"Unknown post-treatment feature: {feature_name}")

            prediction = _predict_one_row_with_fit(fit, row)
            predictions.append(prediction)
            history.append(prediction)

        return np.asarray(predictions, dtype=float)

    def _fit_selected_model_and_predict(
        model_name,
        y_all,
        X_all_current,
        fit_pre_positions,
        predict_positions,
        post_positions,
        n_pre_total,
        current_feature_names,
        selected_spec,
        standardize_model_data,
        seed,
        predict_post=True,
    ):
        stage_start_time = time.perf_counter()

        lag_construction_start_time = time.perf_counter()

        X_all_donor_lags, X_all_treated_lags, donor_lag_map, treated_lag_names = _make_lag_frames(
            y_all,
            X_all_current,
        )

        lag_construction_seconds = _elapsed_seconds(lag_construction_start_time)

        design_start_time = time.perf_counter()

        prepared = _prepare_design(
            train_positions=fit_pre_positions,
            predict_positions=predict_positions,
            y_all=y_all,
            X_all_current=X_all_current,
            X_all_donor_lags=X_all_donor_lags,
            X_all_treated_lags=X_all_treated_lags,
            current_feature_names=current_feature_names,
            donor_lag_map=donor_lag_map,
            treated_lag_names=treated_lag_names,
            standardize_model_data=standardize_model_data,
        )

        design_seconds = _elapsed_seconds(design_start_time)

        fit_start_time = time.perf_counter()

        fit = _fit_model_from_spec(
            model_name=model_name,
            prepared=prepared,
            spec=selected_spec,
            seed=seed,
            fixed_epochs=selected_spec.get("best_epoch", None),
        )

        fit_seconds = _elapsed_seconds(fit_start_time)

        predict_start_time = time.perf_counter()

        predict_prediction = _predict_with_fit(fit, prepared["X_predict_raw"])

        predict_seconds = _elapsed_seconds(predict_start_time)

        if predict_post:
            post_predict_start_time = time.perf_counter()

            post_prediction = _predict_post_period(
                fit=fit,
                y_all=y_all,
                X_all_current=X_all_current,
                X_all_donor_lags=X_all_donor_lags,
                X_all_treated_lags=X_all_treated_lags,
                post_positions=post_positions,
                n_pre_total=n_pre_total,
            )

            post_predict_seconds = _elapsed_seconds(post_predict_start_time)

        else:
            post_prediction = np.full(len(post_positions), np.nan, dtype=float)
            post_predict_seconds = 0.0

        timing = {
            "lag_construction_seconds": lag_construction_seconds,
            "design_seconds": design_seconds,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "post_predict_seconds": post_predict_seconds,
            "total_seconds": _elapsed_seconds(stage_start_time),
            "predict_post": bool(predict_post),
        }

        return {
            "fit": fit,
            "prepared": prepared,
            "predict_prediction": predict_prediction,
            "post_prediction": post_prediction,
            "X_all_donor_lags": X_all_donor_lags,
            "X_all_treated_lags": X_all_treated_lags,
            "timing": timing,
        }
    
    # ---------------------------------------------------------------------
    # Cross validation
    # ---------------------------------------------------------------------
    def _make_blocked_cv_folds(positions, number_of_folds):
        positions = np.asarray(positions, dtype=int)
        number_of_folds = min(int(number_of_folds), len(positions))
        fold_blocks = [fold for fold in np.array_split(positions, number_of_folds) if len(fold) > 0]

        folds = []

        for validation_fold in fold_blocks:
            validation_set = set(validation_fold.tolist())
            training_fold = np.asarray(
                [position for position in positions if position not in validation_set],
                dtype=int,
            )

            if len(training_fold) >= 2 and len(validation_fold) >= 1:
                folds.append((training_fold, validation_fold))

        return folds

    def _candidate_specs_for_model(model_name):
        specs = []

        if model_name == "SCM":
            for current_positive_weights in positive_weights_grid:
                for current_sum_to_one in sum_to_one_grid_local:
                    for current_lambda in lambda_grid:
                        specs.append(
                            {
                                "model": "SCM",
                                "lambda": float(current_lambda),
                                "sum_to_one_constraint": bool(current_sum_to_one),
                                "positive_weights_constraint": bool(current_positive_weights),
                                "best_epoch": np.nan,
                            }
                        )

        elif model_name == "NN-SCM":
            for current_lambda in nn_lambda_grid:
                specs.append(
                    {
                        "model": "NN-SCM",
                        "lambda": float(current_lambda),
                        "sum_to_one_constraint": np.nan,
                        "positive_weights_constraint": np.nan,
                        "best_epoch": np.nan,
                    }
                )

        else:
            raise ValueError(f"Unknown model name: {model_name}")

        return specs

    def _select_with_single_validation_split(
        dataset_key,
        dataset_number,
        model_name,
        y_all,
        X_all_current,
        X_all_donor_lags,
        X_all_treated_lags,
        donor_lag_map,
        treated_lag_names,
        selected_controls,
        training_positions,
        validation_positions,
        metric_scale_value,
        standardize_model_data,
    ):
        design_start_time = time.perf_counter()

        prepared_train_validation = _prepare_design(
            train_positions=training_positions,
            predict_positions=validation_positions,
            y_all=y_all,
            X_all_current=X_all_current,
            X_all_donor_lags=X_all_donor_lags,
            X_all_treated_lags=X_all_treated_lags,
            current_feature_names=selected_controls,
            donor_lag_map=donor_lag_map,
            treated_lag_names=treated_lag_names,
            standardize_model_data=standardize_model_data,
        )

        shared_selection_design_seconds = _elapsed_seconds(design_start_time)

        selection_rows = []

        for candidate_number, spec in enumerate(_candidate_specs_for_model(model_name)):
            candidate_start_time = time.perf_counter()

            fit_start_time = time.perf_counter()

            fit = _fit_model_from_spec(
                model_name=model_name,
                prepared=prepared_train_validation,
                spec=spec,
                seed=int(random_state) + 10_000 + dataset_number * 1000 + candidate_number,
                fixed_epochs=None,
            )

            selection_fit_seconds = _elapsed_seconds(fit_start_time)

            training_inference_start_time = time.perf_counter()

            training_prediction = _predict_with_fit(
                fit,
                prepared_train_validation["X_train_raw"],
            )

            selection_training_inference_seconds = _elapsed_seconds(training_inference_start_time)

            validation_inference_start_time = time.perf_counter()

            validation_prediction = _predict_with_fit(
                fit,
                prepared_train_validation["X_predict_raw"],
            )

            selection_validation_inference_seconds = _elapsed_seconds(validation_inference_start_time)

            row = {
                "dataset": dataset_key,
                "model": model_name,
                "selection_mode": "single_validation_split",
                "lambda": float(spec["lambda"]),
                "sum_to_one_constraint": spec["sum_to_one_constraint"],
                "positive_weights_constraint": spec["positive_weights_constraint"],
                "best_epoch": int(fit["best_epoch"]) if model_name == "NN-SCM" else np.nan,
                "n_features": int(len(fit["feature_names"])),
                "metric_scale": float(metric_scale_value),

                "selection_design_seconds": shared_selection_design_seconds,
                "selection_fit_seconds": selection_fit_seconds,
                "selection_training_inference_seconds": selection_training_inference_seconds,
                "selection_validation_inference_seconds": selection_validation_inference_seconds,
                "selection_total_inference_seconds": (
                    selection_training_inference_seconds
                    + selection_validation_inference_seconds
                ),
                "selection_total_seconds": _elapsed_seconds(candidate_start_time),
            }

            row.update(
                _error_metrics(
                    prepared_train_validation["y_train_raw"],
                    training_prediction,
                    "training",
                    metric_scale_value,
                )
            )

            row.update(
                _error_metrics(
                    prepared_train_validation["y_predict_raw"],
                    validation_prediction,
                    "validation",
                    metric_scale_value,
                )
            )

            selection_rows.append(row)

        selection_df = pd.DataFrame(selection_rows)

        best_row = (
            selection_df
            .sort_values(
                [
                    "validation_scaled_mse",
                    "validation_scaled_mae",
                    "lambda",
                    "sum_to_one_constraint",
                    "positive_weights_constraint",
                ],
                na_position="last",
            )
            .iloc[0]
        )

        return best_row, selection_df

    def _select_with_blocked_cross_validation(
        dataset_key,
        dataset_number,
        model_name,
        y_all,
        X_all_current,
        selected_controls,
        cv_positions,
        metric_scale_value,
        standardize_model_data,
    ):
        folds = _make_blocked_cv_folds(cv_positions, cv_folds)

        if len(folds) < 2:
            raise ValueError(
                f"{dataset_key}: Cross validation needs at least two valid folds. "
                f"Got {len(folds)} fold."
            )

        selection_rows = []

        for candidate_number, spec in enumerate(_candidate_specs_for_model(model_name)):
            candidate_start_time = time.perf_counter()

            fold_rows = []
            fold_best_epochs = []
            fold_timing_rows = []
            fold_feature_counts = []

            for fold_number, (fold_training_positions, fold_validation_positions) in enumerate(folds):
                fold_total_start_time = time.perf_counter()

                lag_start_time = time.perf_counter()

                X_fold_donor_lags, X_fold_treated_lags, fold_donor_lag_map, fold_treated_lag_names = _make_lag_frames(
                    y_all,
                    X_all_current,
                )

                fold_lag_construction_seconds = _elapsed_seconds(lag_start_time)

                design_start_time = time.perf_counter()

                prepared_fold = _prepare_design(
                    train_positions=fold_training_positions,
                    predict_positions=fold_validation_positions,
                    y_all=y_all,
                    X_all_current=X_all_current,
                    X_all_donor_lags=X_fold_donor_lags,
                    X_all_treated_lags=X_fold_treated_lags,
                    current_feature_names=selected_controls,
                    donor_lag_map=fold_donor_lag_map,
                    treated_lag_names=fold_treated_lag_names,
                    standardize_model_data=standardize_model_data,
                )

                fold_design_seconds = _elapsed_seconds(design_start_time)

                fit_start_time = time.perf_counter()

                fit = _fit_model_from_spec(
                    model_name=model_name,
                    prepared=prepared_fold,
                    spec=spec,
                    seed=int(random_state)
                    + 20_000
                    + dataset_number * 1000
                    + candidate_number * 100
                    + fold_number,
                    fixed_epochs=None,
                )

                fold_fit_seconds = _elapsed_seconds(fit_start_time)

                training_inference_start_time = time.perf_counter()

                training_prediction = _predict_with_fit(fit, prepared_fold["X_train_raw"])

                fold_training_inference_seconds = _elapsed_seconds(training_inference_start_time)

                validation_inference_start_time = time.perf_counter()

                validation_prediction = _predict_with_fit(fit, prepared_fold["X_predict_raw"])

                fold_validation_inference_seconds = _elapsed_seconds(validation_inference_start_time)

                fold_metrics = {}

                fold_metrics.update(
                    _error_metrics(
                        prepared_fold["y_train_raw"],
                        training_prediction,
                        "training",
                        metric_scale_value,
                    )
                )

                fold_metrics.update(
                    _error_metrics(
                        prepared_fold["y_predict_raw"],
                        validation_prediction,
                        "validation",
                        metric_scale_value,
                    )
                )

                fold_rows.append(fold_metrics)
                fold_feature_counts.append(int(len(fit["feature_names"])))

                fold_timing_rows.append(
                    {
                        "lag_construction_seconds": fold_lag_construction_seconds,
                        "design_seconds": fold_design_seconds,
                        "fit_seconds": fold_fit_seconds,
                        "training_inference_seconds": fold_training_inference_seconds,
                        "validation_inference_seconds": fold_validation_inference_seconds,
                        "total_inference_seconds": (
                            fold_training_inference_seconds
                            + fold_validation_inference_seconds
                        ),
                        "total_seconds": _elapsed_seconds(fold_total_start_time),
                    }
                )

                if model_name == "NN-SCM":
                    fold_best_epochs.append(int(fit["best_epoch"]))

            fold_df = pd.DataFrame(fold_rows)
            fold_timing_df = pd.DataFrame(fold_timing_rows)

            row = {
                "dataset": dataset_key,
                "model": model_name,
                "selection_mode": "blocked_k_fold_cv",
                "cv_folds": int(len(folds)),
                "lambda": float(spec["lambda"]),
                "sum_to_one_constraint": spec["sum_to_one_constraint"],
                "positive_weights_constraint": spec["positive_weights_constraint"],
                "best_epoch": (
                    int(np.median(fold_best_epochs))
                    if model_name == "NN-SCM" and len(fold_best_epochs) > 0
                    else np.nan
                ),
                "n_features": float(np.mean(fold_feature_counts)) if len(fold_feature_counts) > 0 else np.nan,
                "n_features_min": int(np.min(fold_feature_counts)) if len(fold_feature_counts) > 0 else np.nan,
                "n_features_max": int(np.max(fold_feature_counts)) if len(fold_feature_counts) > 0 else np.nan,
                "metric_scale": float(metric_scale_value),

                "selection_lag_construction_seconds_sum": float(fold_timing_df["lag_construction_seconds"].sum()),
                "selection_lag_construction_seconds_mean": float(fold_timing_df["lag_construction_seconds"].mean()),

                "selection_design_seconds_sum": float(fold_timing_df["design_seconds"].sum()),
                "selection_design_seconds_mean": float(fold_timing_df["design_seconds"].mean()),

                "selection_fit_seconds": float(fold_timing_df["fit_seconds"].sum()),
                "selection_fit_seconds_mean": float(fold_timing_df["fit_seconds"].mean()),

                "selection_training_inference_seconds": float(fold_timing_df["training_inference_seconds"].sum()),
                "selection_training_inference_seconds_mean": float(fold_timing_df["training_inference_seconds"].mean()),

                "selection_validation_inference_seconds": float(fold_timing_df["validation_inference_seconds"].sum()),
                "selection_validation_inference_seconds_mean": float(fold_timing_df["validation_inference_seconds"].mean()),

                "selection_total_inference_seconds": float(fold_timing_df["total_inference_seconds"].sum()),
                "selection_total_inference_seconds_mean": float(fold_timing_df["total_inference_seconds"].mean()),

                "selection_fold_total_seconds_sum": float(fold_timing_df["total_seconds"].sum()),
                "selection_fold_total_seconds_mean": float(fold_timing_df["total_seconds"].mean()),
                "selection_total_seconds": _elapsed_seconds(candidate_start_time),
            }

            metric_columns = [
                column for column in fold_df.columns
                if (
                    column.startswith("training_")
                    or column.startswith("validation_")
                )
            ]

            for column in metric_columns:
                numeric_values = pd.to_numeric(fold_df[column], errors="coerce")

                if numeric_values.notna().any():
                    row[column] = float(numeric_values.mean())

            selection_rows.append(row)

        selection_df = pd.DataFrame(selection_rows)

        best_row = (
            selection_df
            .sort_values(
                [
                    "validation_scaled_mse",
                    "validation_scaled_mae",
                    "lambda",
                    "sum_to_one_constraint",
                    "positive_weights_constraint",
                ],
                na_position="last",
            )
            .iloc[0]
        )

        return best_row, selection_df
    
    # ---------------------------------------------------------------------
    # Weights and interpretability
    # ---------------------------------------------------------------------
    def _build_weights_df(fit):
        if fit["model_type"] != "SCM":
            return pd.DataFrame(
                columns=[
                    "model",
                    "lambda",
                    "sum_to_one_constraint",
                    "positive_weights_constraint",
                    "feature_name",
                    "feature_type",
                    "donor_name",
                    "weight",
                    "abs_weight",
                    "abs_weight_share",
                    "abs_weight_rank",
                ]
            )

        weights = np.asarray(fit["weights"], dtype=float)
        abs_weights = np.abs(weights)
        abs_weight_sum = float(abs_weights.sum())

        order = np.argsort(-abs_weights)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(order) + 1)

        weights_df = pd.DataFrame(
            {
                "model": fit["model_type"],
                "lambda": fit["lambda"],
                "sum_to_one_constraint": fit["sum_to_one_constraint"],
                "positive_weights_constraint": fit["positive_weights_constraint"],
                "feature_name": fit["feature_names"],
                "feature_type": [_classify_feature(name) for name in fit["feature_names"]],
                "donor_name": [_extract_donor_name_from_feature(name) for name in fit["feature_names"]],
                "weight": weights,
                "abs_weight": abs_weights,
                "abs_weight_share": abs_weights / abs_weight_sum if abs_weight_sum > 0 else np.zeros_like(abs_weights),
                "abs_weight_rank": ranks,
            }
        )

        return weights_df.sort_values("abs_weight", ascending=False).reset_index(drop=True)

    def _effective_number_from_abs_weights(abs_values):
        abs_values = _as_1d(abs_values)
        abs_sum = float(np.sum(abs_values))

        return _safe_divide(
            abs_sum ** 2,
            float(np.sum(abs_values ** 2)),
        )

    def _add_weight_diagnostics(metrics, weights_df, n_used_current_controls):
        """
        Report donor support twice.

        Without lags:
            Uses only current donor features.

        With lags:
            Aggregates current donor features and donor-lag features by donor.
            Treated-unit lag features are excluded from this donor count.
        """
        if weights_df.empty:
            metrics["effective_donor_number_without_lags"] = np.nan
            metrics["effective_donor_share_without_lags"] = np.nan
            metrics["negative_weight_share_without_lags"] = np.nan
            metrics["top_donor_abs_weight_share_without_lags"] = np.nan

            metrics["effective_donor_number_with_lags"] = np.nan
            metrics["effective_donor_share_with_lags"] = np.nan
            metrics["negative_weight_share_with_lags"] = np.nan
            metrics["top_donor_abs_weight_share_with_lags"] = np.nan

            metrics["effective_feature_number_all_features"] = np.nan
            metrics["negative_weight_share_all_features"] = np.nan

            metrics["effective_donor_number"] = np.nan
            metrics["effective_donor_share"] = np.nan
            return

        current_weights = weights_df[weights_df["feature_type"] == "current_donor"].copy()
        donor_and_donor_lag_weights = weights_df[
            weights_df["feature_type"].isin(["current_donor", "donor_lag"])
        ].copy()

        all_weights = weights_df.copy()

        current_abs = current_weights["abs_weight"].to_numpy(dtype=float)
        current_abs_sum = float(np.sum(current_abs))

        metrics["effective_donor_number_without_lags"] = _effective_number_from_abs_weights(current_abs)

        metrics["effective_donor_share_without_lags"] = _safe_divide(
            metrics["effective_donor_number_without_lags"],
            int(n_used_current_controls),
        )

        metrics["negative_weight_share_without_lags"] = _safe_divide(
            float(
                current_weights.loc[
                    current_weights["weight"] < 0,
                    "abs_weight",
                ].sum()
            ),
            current_abs_sum,
        )

        metrics["top_donor_abs_weight_share_without_lags"] = _safe_divide(
            float(np.max(current_abs)) if len(current_abs) > 0 else np.nan,
            current_abs_sum,
        )

        donor_grouped_abs = (
            donor_and_donor_lag_weights
            .dropna(subset=["donor_name"])
            .groupby("donor_name")["abs_weight"]
            .sum()
        )

        donor_lag_abs_sum = float(donor_grouped_abs.sum())

        metrics["effective_donor_number_with_lags"] = _effective_number_from_abs_weights(
            donor_grouped_abs.to_numpy(dtype=float)
        )

        metrics["effective_donor_share_with_lags"] = _safe_divide(
            metrics["effective_donor_number_with_lags"],
            int(n_used_current_controls),
        )

        metrics["negative_weight_share_with_lags"] = _safe_divide(
            float(
                donor_and_donor_lag_weights.loc[
                    donor_and_donor_lag_weights["weight"] < 0,
                    "abs_weight",
                ].sum()
            ),
            donor_lag_abs_sum,
        )

        metrics["top_donor_abs_weight_share_with_lags"] = _safe_divide(
            float(donor_grouped_abs.max()) if len(donor_grouped_abs) > 0 else np.nan,
            donor_lag_abs_sum,
        )

        all_abs = all_weights["abs_weight"].to_numpy(dtype=float)
        all_abs_sum = float(np.sum(all_abs))

        metrics["effective_feature_number_all_features"] = _effective_number_from_abs_weights(all_abs)

        metrics["negative_weight_share_all_features"] = _safe_divide(
            float(
                all_weights.loc[
                    all_weights["weight"] < 0,
                    "abs_weight",
                ].sum()
            ),
            all_abs_sum,
        )

        # Backward-compatible aliases.
        metrics["effective_donor_number"] = metrics["effective_donor_number_without_lags"]
        metrics["effective_donor_share"] = metrics["effective_donor_share_without_lags"]

    # ---------------------------------------------------------------------
    # Robustness diagnostics for both SCM and NN-SCM
    # ---------------------------------------------------------------------
    def _add_placebo_p_values(selected_metrics, placebo_results_df):
        selected_metrics["placebo_n"] = int(len(placebo_results_df))

        if placebo_results_df.empty:
            selected_metrics["placebo_p_value_abs_scaled_mean_effect"] = np.nan
            selected_metrics["placebo_p_value_abs_scaled_cumulative_effect"] = np.nan
            selected_metrics["placebo_p_value_post_to_pre_scaled_mse_ratio"] = np.nan
            selected_metrics["placebo_p_value_post_to_pre_scaled_mae_ratio"] = np.nan
            return

        def _empirical_upper_tail_p_value(true_value, placebo_values):
            placebo_values = _as_1d(placebo_values)
            return float((1.0 + np.sum(placebo_values >= true_value)) / (len(placebo_values) + 1.0))

        selected_metrics["placebo_p_value_abs_scaled_mean_effect"] = _empirical_upper_tail_p_value(
            float(selected_metrics["post_abs_scaled_mean_gap"]),
            placebo_results_df["post_abs_scaled_mean_gap"].to_numpy(dtype=float),
        )

        selected_metrics["placebo_p_value_abs_scaled_cumulative_effect"] = _empirical_upper_tail_p_value(
            float(selected_metrics["post_abs_scaled_cumulative_gap"]),
            placebo_results_df["post_abs_scaled_cumulative_gap"].to_numpy(dtype=float),
        )

        selected_metrics["placebo_p_value_post_to_pre_scaled_mse_ratio"] = _empirical_upper_tail_p_value(
            float(selected_metrics["post_to_pre_scaled_mse_ratio"]),
            placebo_results_df["post_to_pre_scaled_mse_ratio"].to_numpy(dtype=float),
        )

        selected_metrics["placebo_p_value_post_to_pre_scaled_mae_ratio"] = _empirical_upper_tail_p_value(
            float(selected_metrics["post_to_pre_scaled_mae_ratio"]),
            placebo_results_df["post_to_pre_scaled_mae_ratio"].to_numpy(dtype=float),
        )

    def _run_placebo_tests_for_model(
        dataset_key,
        dataset_number,
        model_name,
        X_all_current,
        selected_metrics,
        selected_controls,
        pre_positions,
        post_positions,
        n_pre_total,
    ):
        placebo_units = list(selected_controls)

        if max_placebos is not None:
            rng = np.random.default_rng(
                int(random_state) + 40_000 + _stable_int_from_text(dataset_key) % 10_000
            )
            placebo_units = sorted(
                rng.choice(
                    placebo_units,
                    size=min(int(max_placebos), len(placebo_units)),
                    replace=False,
                ).tolist()
            )

        placebo_rows = []

        standardize_model_data = (
            standardize_linear_features
            if model_name == "SCM"
            else standardize_nn_features
        )

        for placebo_number, placebo_unit in enumerate(placebo_units):
            placebo_controls = [
                donor for donor in selected_controls
                if donor != placebo_unit
            ]

            if len(placebo_controls) == 0:
                raise ValueError(
                    f"{dataset_key}: placebo unit {placebo_unit!r} has no remaining donor controls."
                )

            y_placebo = pd.Series(
                X_all_current[placebo_unit].to_numpy(dtype=float),
                index=X_all_current.index,
                name="placebo_actual",
            )

            X_placebo_current = X_all_current[placebo_controls].copy()
            placebo_scale = _safe_scale(y_placebo.loc[pre_positions].to_numpy(dtype=float))

            placebo_fit = _fit_selected_model_and_predict(
                model_name=model_name,
                y_all=y_placebo,
                X_all_current=X_placebo_current,
                fit_pre_positions=pre_positions,
                predict_positions=pre_positions,
                post_positions=post_positions,
                n_pre_total=n_pre_total,
                current_feature_names=placebo_controls,
                selected_spec=selected_metrics,
                standardize_model_data=standardize_model_data,
                seed=int(random_state)
                + 50_000
                + dataset_number * 1000
                + placebo_number,
            )

            placebo_metrics = {
                "dataset": dataset_key,
                "model": model_name,
                "placebo_unit": placebo_unit,
                "n_placebo_controls": int(len(placebo_controls)),
                "n_features": int(len(placebo_fit["fit"]["feature_names"])),
                "lambda": float(selected_metrics["lambda"]),
                "sum_to_one_constraint": selected_metrics.get("sum_to_one_constraint", np.nan),
                "positive_weights_constraint": selected_metrics.get("positive_weights_constraint", np.nan),
                "best_epoch": selected_metrics.get("best_epoch", np.nan),
                "metric_scale": float(placebo_scale),
            }

            if model_name == "SCM":
                placebo_metrics["weight_sum"] = float(np.sum(placebo_fit["fit"]["weights"]))
                placebo_metrics["n_active_weights"] = int(np.sum(np.abs(placebo_fit["fit"]["weights"]) > 1e-8))
            else:
                placebo_metrics["weight_sum"] = np.nan
                placebo_metrics["n_active_weights"] = np.nan

            placebo_metrics.update(
                _error_metrics(
                    y_placebo.loc[pre_positions].to_numpy(dtype=float),
                    placebo_fit["predict_prediction"],
                    "pre",
                    placebo_scale,
                )
            )

            placebo_metrics.update(
                _error_metrics(
                    y_placebo.loc[post_positions].to_numpy(dtype=float),
                    placebo_fit["post_prediction"],
                    "post",
                    placebo_scale,
                )
            )

            _add_post_pre_ratios(placebo_metrics, "pre", "post", "post_to_pre")

            placebo_rows.append(placebo_metrics)

        placebo_results_df = pd.DataFrame(placebo_rows)
        _add_placebo_p_values(selected_metrics, placebo_results_df)

        return placebo_results_df

    def _add_jackknife_summary(selected_metrics, jackknife_results_df):
        selected_metrics["jackknife_n"] = int(len(jackknife_results_df))

        if jackknife_results_df.empty:
            selected_metrics["jackknife_cv_scaled_mean_effect"] = np.nan
            selected_metrics["jackknife_cv_abs_scaled_mean_effect"] = np.nan
            selected_metrics["jackknife_cv_scaled_cumulative_effect"] = np.nan
            selected_metrics["jackknife_cv_abs_scaled_cumulative_effect"] = np.nan
            selected_metrics["jackknife_cv_scaled_mse"] = np.nan
            selected_metrics["jackknife_cv_scaled_mae"] = np.nan
            return

        selected_metrics["jackknife_cv_scaled_mean_effect"] = _coefficient_of_variation(
            jackknife_results_df["post_scaled_mean_gap"].to_numpy(dtype=float)
        )

        selected_metrics["jackknife_cv_abs_scaled_mean_effect"] = _coefficient_of_variation(
            jackknife_results_df["post_abs_scaled_mean_gap"].to_numpy(dtype=float)
        )

        selected_metrics["jackknife_mean_abs_scaled_mean_effect"] = float(
            jackknife_results_df["post_abs_scaled_mean_gap"].mean()
        )

        selected_metrics["jackknife_std_abs_scaled_mean_effect"] = float(
            jackknife_results_df["post_abs_scaled_mean_gap"].std(ddof=0)
        )

        selected_metrics["jackknife_cv_scaled_cumulative_effect"] = _coefficient_of_variation(
            jackknife_results_df["post_scaled_cumulative_gap"].to_numpy(dtype=float)
        )

        selected_metrics["jackknife_cv_abs_scaled_cumulative_effect"] = _coefficient_of_variation(
            jackknife_results_df["post_abs_scaled_cumulative_gap"].to_numpy(dtype=float)
        )

        selected_metrics["jackknife_cv_scaled_mse"] = _coefficient_of_variation(
            jackknife_results_df["post_scaled_mse"].to_numpy(dtype=float)
        )

        selected_metrics["jackknife_cv_scaled_mae"] = _coefficient_of_variation(
            jackknife_results_df["post_scaled_mae"].to_numpy(dtype=float)
        )

    def _run_jackknife_tests_for_model(
        dataset_key,
        dataset_number,
        model_name,
        y_all,
        X_all_current,
        selected_metrics,
        selected_weights,
        selected_controls,
        pre_positions,
        post_positions,
        n_pre_total,
        metric_scale_value,
    ):
        if not selected_weights.empty:
            current_weight_lookup = (
                selected_weights[selected_weights["feature_type"] == "current_donor"]
                .set_index("feature_name")["abs_weight"]
                .to_dict()
            )

            jackknife_candidates = sorted(
                selected_controls,
                key=lambda donor: current_weight_lookup.get(donor, 0.0),
                reverse=True,
            )

        else:
            y_pre = y_all.loc[pre_positions].to_numpy(dtype=float)

            jackknife_candidates = sorted(
                selected_controls,
                key=lambda donor: _safe_abs_corr(
                    y_pre,
                    X_all_current.loc[pre_positions, donor].to_numpy(dtype=float),
                ),
                reverse=True,
            )

        if max_jackknife_donors is not None:
            jackknife_candidates = jackknife_candidates[: int(max_jackknife_donors)]

        jackknife_rows = []

        standardize_model_data = (
            standardize_linear_features
            if model_name == "SCM"
            else standardize_nn_features
        )

        for donor_number, leave_out_donor in enumerate(jackknife_candidates):
            jackknife_controls = [
                donor for donor in selected_controls
                if donor != leave_out_donor
            ]

            if len(jackknife_controls) == 0:
                raise ValueError(
                    f"{dataset_key}: jackknife removal {leave_out_donor!r} leaves no controls."
                )

            jackknife_fit = _fit_selected_model_and_predict(
                model_name=model_name,
                y_all=y_all,
                X_all_current=X_all_current[jackknife_controls].copy(),
                fit_pre_positions=pre_positions,
                predict_positions=pre_positions,
                post_positions=post_positions,
                n_pre_total=n_pre_total,
                current_feature_names=jackknife_controls,
                selected_spec=selected_metrics,
                standardize_model_data=standardize_model_data,
                seed=int(random_state)
                + 60_000
                + dataset_number * 1000
                + donor_number,
            )

            jackknife_metrics = {
                "dataset": dataset_key,
                "model": model_name,
                "leave_out_donor": leave_out_donor,
                "n_jackknife_controls": int(len(jackknife_controls)),
                "n_features": int(len(jackknife_fit["fit"]["feature_names"])),
                "lambda": float(selected_metrics["lambda"]),
                "sum_to_one_constraint": selected_metrics.get("sum_to_one_constraint", np.nan),
                "positive_weights_constraint": selected_metrics.get("positive_weights_constraint", np.nan),
                "best_epoch": selected_metrics.get("best_epoch", np.nan),
                "metric_scale": float(metric_scale_value),
            }

            if model_name == "SCM":
                jackknife_metrics["weight_sum"] = float(np.sum(jackknife_fit["fit"]["weights"]))
                jackknife_metrics["n_active_weights"] = int(np.sum(np.abs(jackknife_fit["fit"]["weights"]) > 1e-8))
            else:
                jackknife_metrics["weight_sum"] = np.nan
                jackknife_metrics["n_active_weights"] = np.nan

            jackknife_metrics.update(
                _error_metrics(
                    y_all.loc[pre_positions].to_numpy(dtype=float),
                    jackknife_fit["predict_prediction"],
                    "pre",
                    metric_scale_value,
                )
            )

            jackknife_metrics.update(
                _error_metrics(
                    y_all.loc[post_positions].to_numpy(dtype=float),
                    jackknife_fit["post_prediction"],
                    "post",
                    metric_scale_value,
                )
            )

            _add_post_pre_ratios(jackknife_metrics, "pre", "post", "post_to_pre")

            jackknife_rows.append(jackknife_metrics)

        jackknife_results_df = pd.DataFrame(jackknife_rows)
        _add_jackknife_summary(selected_metrics, jackknife_results_df)

        return jackknife_results_df

    # ---------------------------------------------------------------------
    # Plotting
    # ---------------------------------------------------------------------
    def _smooth_for_plot(values, window):
        return (
            pd.Series(values)
            .rolling(window=int(window), center=True, min_periods=1)
            .mean()
            .to_numpy(dtype=float)
        )

    def _plot_dataset_result(dataset_key, meta, trajectories, selected_model_names):
        if trend_smoothing_window_by_dataset is None:
            smoothing_windows = {
                str(dataset_name).lower(): 300
                for dataset_name in smooth_plot_datasets
            }
        else:
            smoothing_windows = {
                str(key).lower(): value
                for key, value in dict(trend_smoothing_window_by_dataset).items()
            }

        smoothing_window = smoothing_windows.get(str(dataset_key).lower(), None)
        use_smoothing = smoothing_window is not None and int(smoothing_window) > 1

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(plot_width, plot_height),
            sharex=True,
            gridspec_kw={"height_ratios": [2.4, 1.0]},
        )

        observed_values = trajectories["treated_actual"].to_numpy(dtype=float)

        if use_smoothing:
            axes[0].plot(
                trajectories["time"],
                observed_values,
                alpha=0.18,
                linewidth=0.7,
                label="Observed raw",
            )
            axes[0].plot(
                trajectories["time"],
                _smooth_for_plot(observed_values, smoothing_window),
                linewidth=2.2,
                label=f"Observed rolling mean {smoothing_window}",
            )
        else:
            axes[0].plot(
                trajectories["time"],
                observed_values,
                linewidth=2.0,
                label="Observed",
            )

        for model_name in selected_model_names:
            suffix = model_name.lower().replace("-", "_")
            synthetic_col = f"synthetic_{suffix}"
            gap_col = f"gap_{suffix}"

            synthetic_values = trajectories[synthetic_col].to_numpy(dtype=float)
            gap_values = trajectories[gap_col].to_numpy(dtype=float)

            if use_smoothing:
                axes[0].plot(
                    trajectories["time"],
                    synthetic_values,
                    alpha=0.18,
                    linewidth=0.7,
                    label=f"{model_name} raw",
                )
                axes[0].plot(
                    trajectories["time"],
                    _smooth_for_plot(synthetic_values, smoothing_window),
                    linewidth=2.2,
                    label=f"{model_name} rolling mean {smoothing_window}",
                )
                axes[1].plot(
                    trajectories["time"],
                    gap_values,
                    alpha=0.18,
                    linewidth=0.7,
                )
                axes[1].plot(
                    trajectories["time"],
                    _smooth_for_plot(gap_values, smoothing_window),
                    linewidth=1.8,
                    label=f"{model_name} gap",
                )
            else:
                axes[0].plot(
                    trajectories["time"],
                    synthetic_values,
                    linewidth=1.9,
                    label=model_name,
                )
                axes[1].plot(
                    trajectories["time"],
                    gap_values,
                    linewidth=1.5,
                    label=f"{model_name} gap",
                )

        axes[0].axvline(
            meta["plot_intervention_time"],
            color="black",
            linestyle="--",
            linewidth=1.1,
            alpha=0.9,
        )
        axes[1].axvline(
            meta["plot_intervention_time"],
            color="black",
            linestyle="--",
            linewidth=1.1,
            alpha=0.9,
        )
        axes[1].axhline(
            0.0,
            color="black",
            linestyle="--",
            linewidth=0.9,
            alpha=0.85,
        )

        axes[0].set_title(
            f"{dataset_key} | observed vs synthetic",
            fontsize=12,
            fontweight="bold",
        )
        axes[0].set_ylabel(str(meta.get("outcome_variable", "outcome")))
        axes[0].grid(True, alpha=0.22)
        axes[0].legend(frameon=True, fontsize=8, ncol=3)

        axes[1].set_title("Gap: observed - synthetic", fontsize=10, fontweight="bold")
        axes[1].set_xlabel(str(meta.get("time_variable", "time")))
        axes[1].set_ylabel("Gap")
        axes[1].grid(True, alpha=0.22)
        axes[1].legend(frameon=True, fontsize=8, ncol=3)

        if pd.api.types.is_datetime64_any_dtype(pd.Index(trajectories["time"])):
            fig.autofmt_xdate()

        plt.tight_layout()
        plt.show()

    def _plot_robustness_diagnostics(dataset_key, selected_metrics_by_model, placebo_results_df, jackknife_results_df):
        models_to_plot = []

        for model_name in selected_metrics_by_model:
            has_placebo = (not placebo_results_df.empty) and (model_name in placebo_results_df["model"].unique())
            has_jackknife = (not jackknife_results_df.empty) and (model_name in jackknife_results_df["model"].unique())

            if has_placebo or has_jackknife:
                models_to_plot.append(model_name)

        if len(models_to_plot) == 0:
            return

        fig, axes = plt.subplots(
            nrows=len(models_to_plot),
            ncols=2,
            figsize=(plot_width, 2.8 * len(models_to_plot)),
            squeeze=False,
        )

        for row_idx, model_name in enumerate(models_to_plot):
            metrics = selected_metrics_by_model[model_name]

            ax_placebo = axes[row_idx, 0]
            ax_jackknife = axes[row_idx, 1]

            model_placebos = (
                placebo_results_df[placebo_results_df["model"] == model_name].copy()
                if not placebo_results_df.empty
                else pd.DataFrame()
            )

            model_jackknife = (
                jackknife_results_df[jackknife_results_df["model"] == model_name].copy()
                if not jackknife_results_df.empty
                else pd.DataFrame()
            )

            if not model_placebos.empty:
                values = np.sort(model_placebos["post_abs_scaled_mean_gap"].to_numpy(dtype=float))

                ax_placebo.scatter(
                    np.arange(len(values)),
                    values,
                    alpha=0.65,
                    s=18,
                    label="Placebos",
                )
                ax_placebo.axhline(
                    float(metrics["post_abs_scaled_mean_gap"]),
                    color="black",
                    linewidth=2.0,
                    linestyle="--",
                    label="Treated",
                )
                ax_placebo.set_title(
                    f"{model_name} placebo | p={_fmt(metrics.get('placebo_p_value_abs_scaled_mean_effect'))}",
                    fontsize=10,
                    fontweight="bold",
                )
                ax_placebo.set_xlabel("Sorted placebo units")
                ax_placebo.set_ylabel("Abs. scaled mean post effect")
                ax_placebo.grid(True, alpha=0.22)
                ax_placebo.legend(frameon=True, fontsize=8)
            else:
                ax_placebo.axis("off")

            if not model_jackknife.empty:
                values = model_jackknife["post_abs_scaled_mean_gap"].to_numpy(dtype=float)

                ax_jackknife.boxplot(
                    values,
                    vert=True,
                    widths=0.4,
                    showmeans=True,
                )
                ax_jackknife.scatter(
                    np.ones(len(values)),
                    values,
                    alpha=0.45,
                    s=18,
                    label="Leave-one-out",
                )
                ax_jackknife.axhline(
                    float(metrics["post_abs_scaled_mean_gap"]),
                    color="black",
                    linewidth=2.0,
                    linestyle="--",
                    label="Selected model",
                )
                ax_jackknife.set_title(
                    f"{model_name} jackknife | CV={_fmt(metrics.get('jackknife_cv_abs_scaled_mean_effect'))}",
                    fontsize=10,
                    fontweight="bold",
                )
                ax_jackknife.set_xticks([1])
                ax_jackknife.set_xticklabels(["Jackknife"])
                ax_jackknife.set_ylabel("Abs. scaled mean post effect")
                ax_jackknife.grid(True, axis="y", alpha=0.22)
                ax_jackknife.legend(frameon=True, fontsize=8)
            else:
                ax_jackknife.axis("off")

        fig.suptitle(
            f"{dataset_key} | robustness diagnostics",
            fontsize=12,
            fontweight="bold",
        )
        plt.tight_layout()
        plt.show()

    # ---------------------------------------------------------------------
    # Summary output
    # ---------------------------------------------------------------------
    def _make_diagnostic_summary(metrics_df):
        summary_columns = [
            "dataset",
            "kind",
            "model",
            "selection_mode",
            "treated_unit",
            "n_used_current_controls",
            "n_features",
            "lambda",
            "sum_to_one_constraint",
            "positive_weights_constraint",
            "best_epoch",

            "training_scaled_mse",
            "training_scaled_mae",
            "validation_scaled_mse",
            "validation_scaled_mae",
            "test_scaled_mse",
            "test_scaled_mae",
            "test_train_scaled_mse_ratio",

            "pre_scaled_mse",
            "pre_scaled_mae",
            "pre_scaled_mean_gap",
            "pre_gap_trend_scaled_total",

            "post_scaled_mse",
            "post_scaled_mae",
            "post_mean_gap",
            "post_scaled_mean_gap",
            "post_cumulative_gap",
            "post_scaled_cumulative_gap",
            "post_same_sign_gap_share",
            "post_scaled_mse_to_mae_ratio",

            "selection_fit_seconds",
            "selection_total_inference_seconds",
            "selection_total_seconds",

            "test_lag_construction_seconds",
            "test_design_seconds",
            "test_refit_seconds",
            "test_inference_seconds",
            "test_post_inference_seconds_unused",
            "test_stage_total_seconds",

            "pre_lag_construction_seconds",
            "pre_design_seconds",
            "pre_refit_seconds",
            "pre_inference_seconds",
            "post_inference_seconds",
            "pre_post_stage_total_seconds",
            "selected_model_total_seconds",

            "placebo_n",
            "placebo_p_value_abs_scaled_mean_effect",
            "placebo_p_value_post_to_pre_scaled_mse_ratio",
            "jackknife_n",
            "jackknife_cv_abs_scaled_mean_effect",

            "effective_donor_number_without_lags",
            "effective_donor_share_without_lags",
            "effective_donor_number_with_lags",
            "effective_donor_share_with_lags",
            "negative_weight_share_without_lags",
            "negative_weight_share_with_lags",
            "effective_feature_number_all_features",

            "post_to_pre_scaled_mse_ratio",
            "post_to_pre_scaled_mae_ratio",
        ]

        return metrics_df.reindex(columns=summary_columns)

    def _print_dataset_summary(dataset_key, selected_metrics_by_model):
        for model_name, metrics in selected_metrics_by_model.items():
            epoch_text = ""
            if model_name == "NN-SCM":
                epoch_text = f" | epoch={_fmt(metrics.get('best_epoch'), 0)}/{int(nn_epochs)}"

            print(
                f"{dataset_key} | {model_name} | "
                f"scaled MSE/MAE train={_fmt(metrics.get('training_scaled_mse'))}/{_fmt(metrics.get('training_scaled_mae'))}, "
                f"val={_fmt(metrics.get('validation_scaled_mse'))}/{_fmt(metrics.get('validation_scaled_mae'))}, "
                f"test={_fmt(metrics.get('test_scaled_mse'))}/{_fmt(metrics.get('test_scaled_mae'))} | "
                f"test/train MSE={_fmt(metrics.get('test_train_scaled_mse_ratio'))} | "
                f"post mean gap={_fmt(metrics.get('post_scaled_mean_gap'))} scaled | "
                f"same-sign={_fmt(metrics.get('post_same_sign_gap_share'))} | "
                f"placebo p={_fmt(metrics.get('placebo_p_value_abs_scaled_mean_effect'))} | "
                f"jackknife CV={_fmt(metrics.get('jackknife_cv_abs_scaled_mean_effect'))} | "
                f"eff donors no-lag/with-lag="
                f"{_fmt(metrics.get('effective_donor_number_without_lags'))}/"
                f"{_fmt(metrics.get('effective_donor_number_with_lags'))}"
                f"{epoch_text}"
            )

    # ---------------------------------------------------------------------
    # Main loop
    # ---------------------------------------------------------------------
    all_results = {}
    all_selected_metric_rows = []

    for dataset_number, dataset_key in enumerate(datasets):
        out = scm_data[dataset_key]
        meta = dict(out["meta"])

        all_controls = [str(control) for control in meta["controls"]]
        n_all_controls = len(all_controls)

        requested_control_group_size = control_group_size

        if control_group_size is None:
            selected_controls = all_controls
            effective_control_group_size = n_all_controls
        else:
            effective_control_group_size = min(int(control_group_size), n_all_controls)

            rng = np.random.default_rng(int(random_state) + dataset_number)
            selected_controls = sorted(
                rng.choice(
                    all_controls,
                    size=effective_control_group_size,
                    replace=False,
                ).tolist()
            )

        selected_control_indices = [all_controls.index(control) for control in selected_controls]

        Z0 = np.asarray(out["Z0"], dtype=float)[:, selected_control_indices]
        Y0 = np.asarray(out["Y0"], dtype=float)[:, selected_control_indices]
        Z1 = _as_1d(out["Z1"])
        Y1 = _as_1d(out["Y1"])

        n_pre_total = Z0.shape[0]
        n_post = Y0.shape[0]
        n_total = n_pre_total + n_post

        time_pre = pd.Index(meta["time_index_pre"])
        time_post = pd.Index(meta["time_index_post"])
        time_all = time_pre.append(time_post)

        meta["plot_intervention_time"] = time_post[0]

        y_all = pd.Series(
            np.concatenate([Z1, Y1]),
            index=np.arange(n_total),
            name="treated_actual",
        )

        X_all_current = pd.DataFrame(
            np.vstack([Z0, Y0]),
            index=np.arange(n_total),
            columns=selected_controls,
        )

        initial_drop_rows = 0

        if use_lagged_treated:
            initial_drop_rows = max(initial_drop_rows, int(lagged_treated_steps))

        if use_topk_donor_lags:
            initial_drop_rows = max(initial_drop_rows, int(donor_lag_past_steps))

        effective_pre_positions = np.arange(initial_drop_rows, n_pre_total)
        post_positions = np.arange(n_pre_total, n_total)

        n_effective_pre = len(effective_pre_positions)

        n_validation = int(np.floor(n_effective_pre * float(pre_validation_fraction)))
        n_test = int(np.floor(n_effective_pre * float(pre_test_fraction)))

        if float(pre_validation_fraction) > 0:
            n_validation = max(1, n_validation)

        if float(pre_test_fraction) > 0:
            n_test = max(1, n_test)

        use_cv_selection = (
            bool(use_cross_validation_if_small)
            and (
                n_validation < int(min_validation_rows)
                or n_test < int(min_test_rows)
            )
        )

        if use_cv_selection:
            n_test = max(n_test, min(int(min_test_rows), max(1, n_effective_pre - 3)))
            n_test = min(n_test, max(1, n_effective_pre - 3))

            train_validation_positions = effective_pre_positions[:-n_test]
            test_positions = effective_pre_positions[-n_test:]

            training_positions = train_validation_positions
            validation_positions = np.asarray([], dtype=int)

            if len(train_validation_positions) < 3:
                raise ValueError(
                    f"{dataset_key}: Not enough pre-treatment rows for cross validation. "
                    f"effective pre={n_effective_pre}, test={len(test_positions)}"
                )

            selection_mode = "blocked_k_fold_cv"

        else:
            n_training = n_effective_pre - n_validation - n_test

            if n_training < 2:
                raise ValueError(
                    f"{dataset_key}: Not enough effective pre-treatment rows. "
                    f"effective pre={n_effective_pre}, training={n_training}, "
                    f"validation={n_validation}, test={n_test}"
                )

            training_positions = effective_pre_positions[:n_training]
            validation_positions = effective_pre_positions[n_training: n_training + n_validation]
            test_positions = effective_pre_positions[n_training + n_validation:]
            train_validation_positions = effective_pre_positions[: n_training + n_validation]

            selection_mode = "single_validation_split"

        pre_positions = effective_pre_positions

        metric_scale_value = _safe_scale(y_all.loc[pre_positions].to_numpy(dtype=float))
        seasonal_period = _infer_seasonal_period(time_pre[initial_drop_rows:], dataset_key)

        X_all_donor_lags, X_all_treated_lags, donor_lag_map, treated_lag_names = _make_lag_frames(
            y_all,
            X_all_current,
        )

        model_selection_frames = []
        selected_fits = {}
        selected_metrics_by_model = {}
        selected_weights_by_model = {}
        selected_model_names = []

        enabled_model_names = []
        if enable_linear_scm:
            enabled_model_names.append("SCM")
        if enable_nn_scm:
            enabled_model_names.append("NN-SCM")

        for model_name in enabled_model_names:
            standardize_model_data = (
                standardize_linear_features
                if model_name == "SCM"
                else standardize_nn_features
            )

            if selection_mode == "blocked_k_fold_cv":
                best_row, selection_df = _select_with_blocked_cross_validation(
                    dataset_key=dataset_key,
                    dataset_number=dataset_number,
                    model_name=model_name,
                    y_all=y_all,
                    X_all_current=X_all_current,
                    selected_controls=selected_controls,
                    cv_positions=train_validation_positions,
                    metric_scale_value=metric_scale_value,
                    standardize_model_data=standardize_model_data,
                )
            else:
                best_row, selection_df = _select_with_single_validation_split(
                    dataset_key=dataset_key,
                    dataset_number=dataset_number,
                    model_name=model_name,
                    y_all=y_all,
                    X_all_current=X_all_current,
                    X_all_donor_lags=X_all_donor_lags,
                    X_all_treated_lags=X_all_treated_lags,
                    donor_lag_map=donor_lag_map,
                    treated_lag_names=treated_lag_names,
                    selected_controls=selected_controls,
                    training_positions=training_positions,
                    validation_positions=validation_positions,
                    metric_scale_value=metric_scale_value,
                    standardize_model_data=standardize_model_data,
                )

            model_selection_frames.append(selection_df)

            selected_spec = best_row.to_dict()

            # Fit on training plus validation, evaluate on test.
            test_fit = _fit_selected_model_and_predict(
                model_name=model_name,
                y_all=y_all,
                X_all_current=X_all_current,
                fit_pre_positions=train_validation_positions,
                predict_positions=test_positions,
                post_positions=post_positions,
                n_pre_total=n_pre_total,
                current_feature_names=selected_controls,
                selected_spec=selected_spec,
                standardize_model_data=standardize_model_data,
                seed=int(random_state) + 30_000 + dataset_number,
                predict_post=False,
            )
            test_timing = test_fit["timing"]

            test_prediction = test_fit["predict_prediction"]

            # Fit on full pre-treatment period and predict pre plus post.
            pre_fit = _fit_selected_model_and_predict(
                model_name=model_name,
                y_all=y_all,
                X_all_current=X_all_current,
                fit_pre_positions=pre_positions,
                predict_positions=pre_positions,
                post_positions=post_positions,
                n_pre_total=n_pre_total,
                current_feature_names=selected_controls,
                selected_spec=selected_spec,
                standardize_model_data=standardize_model_data,
                seed=int(random_state) + 40_000 + dataset_number,
                predict_post=True,
            )
            pre_timing = pre_fit["timing"]

            fit_pre = pre_fit["fit"]
            prepared_pre = pre_fit["prepared"]
            pre_prediction = pre_fit["predict_prediction"]
            post_prediction = pre_fit["post_prediction"]

            weights_df = _build_weights_df(fit_pre)

            metrics = {
                "dataset": dataset_key,
                "kind": meta["kind"],
                "model": model_name,
                "selection_mode": selection_mode,
                "treated_unit": meta["treated_unit"],
                "n_all_controls": int(n_all_controls),
                "n_used_current_controls": int(len(selected_controls)),
                "requested_control_group_size": (
                    np.nan if requested_control_group_size is None else int(requested_control_group_size)
                ),
                "effective_control_group_size": int(effective_control_group_size),
                "n_features": int(len(fit_pre["feature_names"])),
                "n_effective_pre": int(n_effective_pre),
                "n_dropped_initial_pre": int(initial_drop_rows),
                "n_training": int(len(training_positions)),
                "n_validation": int(len(validation_positions)),
                "n_test": int(len(test_positions)),
                "n_post": int(len(post_positions)),
                "lambda": float(best_row["lambda"]),
                "sum_to_one_constraint": best_row.get("sum_to_one_constraint", np.nan),
                "positive_weights_constraint": best_row.get("positive_weights_constraint", np.nan),
                "standardize_features": bool(standardize_model_data),
                "metric_scale": float(metric_scale_value),
                "best_epoch": (
                    int(best_row["best_epoch"])
                    if model_name == "NN-SCM" and pd.notna(best_row["best_epoch"])
                    else np.nan
                ),
                "test_lag_construction_seconds": test_timing["lag_construction_seconds"],
                "test_design_seconds": test_timing["design_seconds"],
                "test_refit_seconds": test_timing["fit_seconds"],
                "test_inference_seconds": test_timing["predict_seconds"],
                "test_post_inference_seconds_unused": test_timing["post_predict_seconds"],
                "test_stage_total_seconds": test_timing["total_seconds"],

                "pre_lag_construction_seconds": pre_timing["lag_construction_seconds"],
                "pre_design_seconds": pre_timing["design_seconds"],
                "pre_refit_seconds": pre_timing["fit_seconds"],
                "pre_inference_seconds": pre_timing["predict_seconds"],
                "post_inference_seconds": pre_timing["post_predict_seconds"],
                "pre_post_stage_total_seconds": pre_timing["total_seconds"],

                "selected_model_total_seconds": (
                    test_timing["total_seconds"]
                    + pre_timing["total_seconds"]
                ),
            }

            # Store training and validation metrics from the selection stage.
            for column in best_row.index:
                if column.startswith("training_") or column.startswith("validation_"):
                    metrics[column] = best_row[column]

            # Test metrics after refitting on training plus validation.
            metrics.update(
                _error_metrics(
                    y_all.loc[test_positions].to_numpy(dtype=float),
                    test_prediction,
                    "test",
                    metric_scale_value,
                )
            )

            _add_test_train_ratios(metrics)

            # Full pre-treatment fit metrics.
            metrics.update(
                _error_metrics(
                    prepared_pre["y_predict_raw"],
                    pre_prediction,
                    "pre",
                    metric_scale_value,
                )
            )

            pre_gaps = prepared_pre["y_predict_raw"] - pre_prediction
            _add_pre_gap_trend(
                metrics,
                pre_gaps,
                metric_scale_value,
                seasonal_period,
            )

            # Post-treatment metrics.
            metrics.update(
                _error_metrics(
                    y_all.loc[post_positions].to_numpy(dtype=float),
                    post_prediction,
                    "post",
                    metric_scale_value,
                )
            )

            _add_post_pre_ratios(metrics, "pre", "post", "post_to_pre")
            _add_post_pre_ratios(metrics, "test", "post", "post_to_test")

            if model_name == "SCM":
                metrics["weight_sum"] = float(np.sum(fit_pre["weights"]))
                metrics["min_weight"] = float(np.min(fit_pre["weights"]))
                metrics["max_weight"] = float(np.max(fit_pre["weights"]))
                metrics["n_active_weights"] = int(np.sum(np.abs(fit_pre["weights"]) > 1e-8))
            else:
                metrics["weight_sum"] = np.nan
                metrics["min_weight"] = np.nan
                metrics["max_weight"] = np.nan
                metrics["n_active_weights"] = np.nan

            _add_weight_diagnostics(
                metrics,
                weights_df,
                n_used_current_controls=len(selected_controls),
            )

            selected_fits[model_name] = {
                "fit": fit_pre,
                "pre_prediction": pre_prediction,
                "post_prediction": post_prediction,
            }

            selected_metrics_by_model[model_name] = metrics
            selected_weights_by_model[model_name] = weights_df
            selected_model_names.append(model_name)

        model_selection_results_df = pd.concat(model_selection_frames, ignore_index=True)
        model_selection_results_df["selected_for_model"] = False

        for model_name in selected_model_names:
            selected_metrics = selected_metrics_by_model[model_name]

            mask = (
                (model_selection_results_df["model"] == model_name)
                & (model_selection_results_df["lambda"] == selected_metrics["lambda"])
            )

            if model_name == "SCM":
                mask = (
                    mask
                    & (
                        model_selection_results_df["sum_to_one_constraint"].astype(bool)
                        == bool(selected_metrics["sum_to_one_constraint"])
                    )
                    & (
                        model_selection_results_df["positive_weights_constraint"].astype(bool)
                        == bool(selected_metrics["positive_weights_constraint"])
                    )
                )

            model_selection_results_df.loc[mask, "selected_for_model"] = True

        # -----------------------------------------------------------------
        # Trajectories
        # -----------------------------------------------------------------
        trajectories = pd.DataFrame(
            {
                "time": time_all,
                "treated_actual": y_all.to_numpy(dtype=float),
                "period": ["pre"] * n_pre_total + ["post"] * n_post,
                "evaluation_split": "dropped_pre",
            }
        )

        trajectories.loc[training_positions, "evaluation_split"] = "training"

        if len(validation_positions) > 0:
            trajectories.loc[validation_positions, "evaluation_split"] = "validation"
        else:
            trajectories.loc[train_validation_positions, "evaluation_split"] = "training_cv_pool"

        trajectories.loc[test_positions, "evaluation_split"] = "test"
        trajectories.loc[post_positions, "evaluation_split"] = "post"

        for model_name in selected_model_names:
            suffix = model_name.lower().replace("-", "_")
            prediction = np.full(n_total, np.nan, dtype=float)

            prediction[pre_positions] = selected_fits[model_name]["pre_prediction"]
            prediction[post_positions] = selected_fits[model_name]["post_prediction"]

            trajectories[f"synthetic_{suffix}"] = prediction
            trajectories[f"gap_{suffix}"] = (
                trajectories["treated_actual"]
                - trajectories[f"synthetic_{suffix}"]
            )

        # -----------------------------------------------------------------
        # Robustness diagnostics for all selected models, including NN-SCM.
        # -----------------------------------------------------------------
        placebo_frames = []
        jackknife_frames = []

        if enable_placebo_tests:
            for model_name in selected_model_names:
                placebo_df_model = _run_placebo_tests_for_model(
                    dataset_key=dataset_key,
                    dataset_number=dataset_number,
                    model_name=model_name,
                    X_all_current=X_all_current,
                    selected_metrics=selected_metrics_by_model[model_name],
                    selected_controls=selected_controls,
                    pre_positions=pre_positions,
                    post_positions=post_positions,
                    n_pre_total=n_pre_total,
                )
                placebo_frames.append(placebo_df_model)

        if enable_jackknife_tests:
            for model_name in selected_model_names:
                jackknife_df_model = _run_jackknife_tests_for_model(
                    dataset_key=dataset_key,
                    dataset_number=dataset_number,
                    model_name=model_name,
                    y_all=y_all,
                    X_all_current=X_all_current,
                    selected_metrics=selected_metrics_by_model[model_name],
                    selected_weights=selected_weights_by_model[model_name],
                    selected_controls=selected_controls,
                    pre_positions=pre_positions,
                    post_positions=post_positions,
                    n_pre_total=n_pre_total,
                    metric_scale_value=metric_scale_value,
                )
                jackknife_frames.append(jackknife_df_model)

        placebo_results_df = (
            pd.concat(placebo_frames, ignore_index=True)
            if len(placebo_frames) > 0
            else pd.DataFrame()
        )

        jackknife_results_df = (
            pd.concat(jackknife_frames, ignore_index=True)
            if len(jackknife_frames) > 0
            else pd.DataFrame()
        )

        weight_tables = [
            table
            for table in selected_weights_by_model.values()
            if isinstance(table, pd.DataFrame) and not table.empty
        ]

        weights_df = (
            pd.concat(weight_tables, ignore_index=True)
            if len(weight_tables) > 0
            else pd.DataFrame()
        )

        selected_metrics_df = pd.DataFrame(list(selected_metrics_by_model.values()))

        for metrics in selected_metrics_by_model.values():
            all_selected_metric_rows.append(metrics)

        all_results[dataset_key] = {
            "trajectories": trajectories,
            "weights": weights_df,
            "model_selection": model_selection_results_df,
            "selected_metrics": selected_metrics_df,
            "placebo_results": placebo_results_df,
            "jackknife_results": jackknife_results_df,
            "selected_fits": selected_fits,
        }

        validation_text = (
            f"validation={len(validation_positions)}"
            if selection_mode == "single_validation_split"
            else f"validation=blocked CV with {min(cv_folds, len(train_validation_positions))} folds"
        )

        print(
            f"\n{dataset_key} | controls={len(selected_controls)}/{n_all_controls} | "
            f"effective pre={n_effective_pre} | training={len(training_positions)} | "
            f"{validation_text} | test={len(test_positions)} | post={len(post_positions)}"
        )

        _print_dataset_summary(dataset_key, selected_metrics_by_model)

        if save_results:
            dataset_dir = save_dir / str(dataset_key)
            dataset_dir.mkdir(parents=True, exist_ok=True)

            trajectories.to_csv(dataset_dir / "trajectories.csv", index=False)
            weights_df.to_csv(dataset_dir / "weights.csv", index=False)
            model_selection_results_df.to_csv(dataset_dir / "model_selection.csv", index=False)
            selected_metrics_df.to_csv(dataset_dir / "selected_metrics.csv", index=False)

            if not placebo_results_df.empty:
                placebo_results_df.to_csv(dataset_dir / "placebo_results.csv", index=False)

            if not jackknife_results_df.empty:
                jackknife_results_df.to_csv(dataset_dir / "jackknife_results.csv", index=False)

        if plot_results:
            _plot_dataset_result(dataset_key, meta, trajectories, selected_model_names)

            if plot_robustness_results and (enable_placebo_tests or enable_jackknife_tests):
                _plot_robustness_diagnostics(
                    dataset_key=dataset_key,
                    selected_metrics_by_model=selected_metrics_by_model,
                    placebo_results_df=placebo_results_df,
                    jackknife_results_df=jackknife_results_df,
                )

    # ---------------------------------------------------------------------
    # Final cross-dataset summaries
    # ---------------------------------------------------------------------
    all_selected_metrics_df = (
        pd.DataFrame(all_selected_metric_rows)
        .sort_values(["dataset", "model"])
        .reset_index(drop=True)
    )

    diagnostic_summary_df = (
        _make_diagnostic_summary(all_selected_metrics_df)
        .round(summary_round_digits)
    )

    if save_results:
        all_selected_metrics_df.to_csv(save_dir / "all_selected_metrics.csv", index=False)
        diagnostic_summary_df.to_csv(save_dir / "diagnostic_summary.csv", index=False)

    if compact_summary:
        print("\nDiagnostic summary")
        print(diagnostic_summary_df.to_string(index=False))

    return all_results, all_selected_metrics_df, diagnostic_summary_df

def run_scm_lag_and_control_size_experiment_grid(
    scm_data,
    datasets=None,
    control_group_sizes=(5, 10, 20, None),
    repetitions_per_size=5,
    feature_modes=("standard_donors", "treated_lags", "treated_and_donor_lags"),
    base_random_state=42,
    base_save_dir="results_scm_grid",
    keep_all_results_in_memory=False,

    enable_linear_scm=True,
    enable_nn_scm=True,

    lambda_grid=(0.01,),
    sum_to_one_grid=(True,),
    positive_weights_constraint=(False,),
    standardize_linear_features=False,

    nn_lambda_grid=(0.05,),
    nn_hidden_units=(16,),
    nn_learning_rate=0.001,
    nn_epochs=50,
    nn_batch_size=256,
    nn_early_stopping_patience=10,
    standardize_nn_features=True,

    lagged_treated_steps=1,
    donor_lag_past_steps=1,
    donor_lag_top_k=1,

    pre_validation_fraction=0.2,
    pre_test_fraction=0.1,
    use_cross_validation_if_small=True,
    min_validation_rows=3,
    min_test_rows=3,
    cv_folds=5,

    enable_placebo_tests=False,
    max_placebos=10,
    enable_jackknife_tests=False,
    max_jackknife_donors=10,

    save_results=True,
    plot_results=False,
    compact_summary=False,
    suppress_inner_output=True,
    skip_existing_runs=True,
):
    """
    Run a structured SCM experiment grid and save model-level and grid-level runtime diagnostics.
    """

    from pathlib import Path
    import contextlib
    import io
    import time
    import numpy as np
    import pandas as pd
    import gc

    if datasets is None:
        datasets = list(scm_data.keys())

    if isinstance(datasets, str):
        datasets = [datasets]

    base_save_dir = Path(base_save_dir)

    if save_results:
        base_save_dir.mkdir(parents=True, exist_ok=True)

    allowed_feature_modes = {
        "standard_donors",
        "treated_lags",
        "treated_and_donor_lags",
    }

    unknown_feature_modes = set(feature_modes) - allowed_feature_modes
    if len(unknown_feature_modes) > 0:
        raise ValueError(f"Unknown feature modes: {sorted(unknown_feature_modes)}")

    def _append_metadata_columns(df, metadata):
        metadata_df = pd.DataFrame(
            [metadata],
            index=df.index,
        )

        return pd.concat(
            [
                df.reset_index(drop=True),
                metadata_df.reset_index(drop=True),
            ],
            axis=1,
        )

    def _control_group_label(control_group_size):
        if control_group_size is None:
            return "all"
        return str(int(control_group_size))

    def _feature_mode_flags(feature_mode):
        if feature_mode == "standard_donors":
            return {
                "use_lagged_treated": False,
                "use_topk_donor_lags": False,
            }

        if feature_mode == "treated_lags":
            return {
                "use_lagged_treated": True,
                "use_topk_donor_lags": False,
            }

        if feature_mode == "treated_and_donor_lags":
            return {
                "use_lagged_treated": True,
                "use_topk_donor_lags": True,
            }

        raise ValueError(f"Unknown feature mode: {feature_mode}")

    def _number_of_repetitions(control_group_size):
        if control_group_size is None:
            return 1
        return int(repetitions_per_size)

    def _run_inner_experiment(
        control_group_size,
        run_random_state,
        feature_flags,
        run_save_dir,
    ):
        kwargs = {
            "scm_data": scm_data,
            "datasets": datasets,

            "control_group_size": control_group_size,
            "random_state": run_random_state,

            "enable_linear_scm": enable_linear_scm,
            "enable_nn_scm": enable_nn_scm,

            "lambda_grid": lambda_grid,
            "nn_lambda_grid": nn_lambda_grid,
            "sum_to_one_grid": sum_to_one_grid,
            "positive_weights_constraint": positive_weights_constraint,

            "standardize_linear_features": standardize_linear_features,
            "standardize_nn_features": standardize_nn_features,

            "nn_hidden_units": nn_hidden_units,
            "nn_learning_rate": nn_learning_rate,
            "nn_epochs": nn_epochs,
            "nn_batch_size": nn_batch_size,
            "nn_early_stopping_patience": nn_early_stopping_patience,

            "use_lagged_treated": feature_flags["use_lagged_treated"],
            "lagged_treated_steps": lagged_treated_steps,
            "use_topk_donor_lags": feature_flags["use_topk_donor_lags"],
            "donor_lag_past_steps": donor_lag_past_steps,
            "donor_lag_top_k": donor_lag_top_k,

            "pre_validation_fraction": pre_validation_fraction,
            "pre_test_fraction": pre_test_fraction,
            "use_cross_validation_if_small": use_cross_validation_if_small,
            "min_validation_rows": min_validation_rows,
            "min_test_rows": min_test_rows,
            "cv_folds": cv_folds,

            "enable_placebo_tests": enable_placebo_tests,
            "max_placebos": max_placebos,
            "enable_jackknife_tests": enable_jackknife_tests,
            "max_jackknife_donors": max_jackknife_donors,

            "save_results": save_results,
            "save_dir": str(run_save_dir),
            "plot_results": plot_results,
            "compact_summary": compact_summary,
        }

        if suppress_inner_output:
            with contextlib.redirect_stdout(io.StringIO()):
                return run_scm_experiments_with_diagnostics(**kwargs)

        return run_scm_experiments_with_diagnostics(**kwargs)

    total_experiment_runs = sum(
        _number_of_repetitions(control_group_size)
        for control_group_size in control_group_sizes
    ) * len(feature_modes)

    n_enabled_models = int(enable_linear_scm) + int(enable_nn_scm)
    expected_selected_model_rows = total_experiment_runs * len(datasets) * n_enabled_models

    print(
        f"Running {total_experiment_runs} experiment-grid runs "
        f"with up to {expected_selected_model_rows} selected model rows. "
        f"Main save folder: {base_save_dir}"
    )

    grid_results = {}
    metric_tables = []
    diagnostic_tables = []
    experiment_plan_rows = []
    runtime_rows = []

    run_counter = 0

    for control_group_size in control_group_sizes:
        control_label = _control_group_label(control_group_size)
        current_repetitions = _number_of_repetitions(control_group_size)

        for repetition in range(current_repetitions):
            run_random_state = int(base_random_state) + 10_000 * int(repetition)

            for feature_mode in feature_modes:
                run_counter += 1
                run_start_time = time.perf_counter()

                feature_flags = _feature_mode_flags(feature_mode)

                experiment_id = (
                    f"features={feature_mode}"
                    f"__controls={control_label}"
                    f"__rep={repetition}"
                )

                run_save_dir = base_save_dir / experiment_id

                metadata = {
                    "experiment_id": experiment_id,
                    "feature_mode": feature_mode,
                    "control_group_size": np.nan if control_group_size is None else int(control_group_size),
                    "control_group_size_label": control_label,
                    "repetition": int(repetition),
                    "base_random_state": int(base_random_state),
                    "run_random_state": int(run_random_state),

                    "enable_linear_scm": bool(enable_linear_scm),
                    "enable_nn_scm": bool(enable_nn_scm),

                    "linear_lambda_grid": str(tuple(lambda_grid)),
                    "sum_to_one_grid": str(tuple(sum_to_one_grid)),
                    "positive_weights_constraint_grid": str(tuple(positive_weights_constraint)),
                    "standardize_linear_features": bool(standardize_linear_features),

                    "nn_lambda_grid": str(tuple(nn_lambda_grid)),
                    "nn_hidden_units": str(tuple(nn_hidden_units)),
                    "nn_learning_rate": float(nn_learning_rate),
                    "nn_epochs": int(nn_epochs),
                    "nn_batch_size": int(nn_batch_size),
                    "nn_early_stopping_patience": int(nn_early_stopping_patience),
                    "standardize_nn_features": bool(standardize_nn_features),

                    "use_lagged_treated": bool(feature_flags["use_lagged_treated"]),
                    "lagged_treated_steps": int(lagged_treated_steps) if feature_flags["use_lagged_treated"] else 0,
                    "use_topk_donor_lags": bool(feature_flags["use_topk_donor_lags"]),
                    "donor_lag_past_steps": int(donor_lag_past_steps) if feature_flags["use_topk_donor_lags"] else 0,
                    "donor_lag_top_k": int(donor_lag_top_k) if feature_flags["use_topk_donor_lags"] else 0,

                    "pre_validation_fraction": float(pre_validation_fraction),
                    "pre_test_fraction": float(pre_test_fraction),
                    "use_cross_validation_if_small": bool(use_cross_validation_if_small),
                    "min_validation_rows": int(min_validation_rows),
                    "min_test_rows": int(min_test_rows),
                    "cv_folds": int(cv_folds),

                    "enable_placebo_tests": bool(enable_placebo_tests),
                    "max_placebos": np.nan if max_placebos is None else int(max_placebos),
                    "enable_jackknife_tests": bool(enable_jackknife_tests),
                    "max_jackknife_donors": np.nan if max_jackknife_donors is None else int(max_jackknife_donors),

                    "run_save_dir": str(run_save_dir),
                }

                experiment_plan_rows.append(metadata)

                run_metrics_path = run_save_dir / "all_selected_metrics.csv"
                run_summary_path = run_save_dir / "diagnostic_summary.csv"

                if skip_existing_runs and run_metrics_path.exists() and run_summary_path.exists():
                    print(
                        f"{run_counter}/{total_experiment_runs} | "
                        f"features={feature_mode} | "
                        f"controls={control_label} | "
                        f"rep={repetition} | skipped existing"
                    )

                    all_selected_metrics_df = pd.read_csv(run_metrics_path)
                    diagnostic_summary_df = pd.read_csv(run_summary_path)

                    all_selected_metrics_df = _append_metadata_columns(
                        all_selected_metrics_df,
                        metadata,
                    )

                    diagnostic_summary_df = _append_metadata_columns(
                        diagnostic_summary_df,
                        metadata,
                    )

                    metric_tables.append(all_selected_metrics_df)
                    diagnostic_tables.append(diagnostic_summary_df)

                    runtime_rows.append(
                        {
                            **metadata,
                            "was_skipped": True,
                            "run_total_seconds": 0.0,
                            "run_wall_clock_seconds_including_io": float(time.perf_counter() - run_start_time),
                        }
                    )

                    continue

                print(
                    f"{run_counter}/{total_experiment_runs} | "
                    f"features={feature_mode} | "
                    f"controls={control_label} | "
                    f"rep={repetition}"
                )

                all_results, all_selected_metrics_df, diagnostic_summary_df = _run_inner_experiment(
                    control_group_size=control_group_size,
                    run_random_state=run_random_state,
                    feature_flags=feature_flags,
                    run_save_dir=run_save_dir,
                )

                run_total_seconds = float(time.perf_counter() - run_start_time)

                all_selected_metrics_df = all_selected_metrics_df.copy()
                diagnostic_summary_df = diagnostic_summary_df.copy()

                all_selected_metrics_df = _append_metadata_columns(
                    all_selected_metrics_df,
                    metadata,
                )

                diagnostic_summary_df = _append_metadata_columns(
                    diagnostic_summary_df,
                    metadata,
                )

                metric_tables.append(all_selected_metrics_df)
                diagnostic_tables.append(diagnostic_summary_df)

                runtime_rows.append(
                    {
                        **metadata,
                        "was_skipped": False,
                        "run_total_seconds": run_total_seconds,
                        "run_wall_clock_seconds_including_io": run_total_seconds,
                    }
                )

                if keep_all_results_in_memory:
                    grid_results[experiment_id] = all_results
                else:
                    del all_results
                    gc.collect()

                    if enable_nn_scm:
                        try:
                            from tensorflow import keras
                            keras.backend.clear_session()
                        except Exception:
                            pass

    grid_metrics_df = (
        pd.concat(metric_tables, ignore_index=True)
        if len(metric_tables) > 0
        else pd.DataFrame()
    )

    grid_diagnostic_summary_df = (
        pd.concat(diagnostic_tables, ignore_index=True)
        if len(diagnostic_tables) > 0
        else pd.DataFrame()
    )

    experiment_plan_df = pd.DataFrame(experiment_plan_rows)
    grid_runtime_log_df = pd.DataFrame(runtime_rows)

    sort_columns = [
        "dataset",
        "model",
        "feature_mode",
        "control_group_size_label",
        "repetition",
    ]

    available_metric_sort_columns = [
        column for column in sort_columns
        if column in grid_metrics_df.columns
    ]

    available_diagnostic_sort_columns = [
        column for column in sort_columns
        if column in grid_diagnostic_summary_df.columns
    ]

    runtime_sort_columns = [
        "feature_mode",
        "control_group_size_label",
        "repetition",
    ]

    available_runtime_sort_columns = [
        column for column in runtime_sort_columns
        if column in grid_runtime_log_df.columns
    ]

    if len(available_metric_sort_columns) > 0:
        grid_metrics_df = (
            grid_metrics_df
            .sort_values(available_metric_sort_columns)
            .reset_index(drop=True)
        )

    if len(available_diagnostic_sort_columns) > 0:
        grid_diagnostic_summary_df = (
            grid_diagnostic_summary_df
            .sort_values(available_diagnostic_sort_columns)
            .reset_index(drop=True)
        )

    if len(available_runtime_sort_columns) > 0:
        grid_runtime_log_df = (
            grid_runtime_log_df
            .sort_values(available_runtime_sort_columns)
            .reset_index(drop=True)
        )

    if save_results:
        grid_metrics_df.to_csv(base_save_dir / "grid_all_selected_metrics.csv", index=False)
        grid_diagnostic_summary_df.to_csv(base_save_dir / "grid_diagnostic_summary.csv", index=False)
        experiment_plan_df.to_csv(base_save_dir / "grid_experiment_plan.csv", index=False)
        grid_runtime_log_df.to_csv(base_save_dir / "grid_runtime_log.csv", index=False)

    print(f"Finished. Aggregated results saved in: {base_save_dir}")

    return (
        grid_results,
        grid_metrics_df,
        grid_diagnostic_summary_df,
        experiment_plan_df,
        grid_runtime_log_df,
    )

# Final execution line
(grid_results,grid_metrics_df,grid_diagnostic_summary_df,experiment_plan_df,grid_runtime_log_df,) = run_scm_lag_and_control_size_experiment_grid(
    scm_data=scm_data, datasets=["bess"],

    control_group_sizes=(2, 50, 100, None), repetitions_per_size=1, feature_modes=("standard_donors", "treated_lags", "treated_and_donor_lags"),

    enable_linear_scm=True, enable_nn_scm=True,
    nn_batch_size=256, nn_epochs=2000, nn_lambda_grid=(0.05,), nn_learning_rate=0.0005, nn_hidden_units=(64, 64),nn_early_stopping_patience=10,

    lagged_treated_steps=336, donor_lag_past_steps=24, donor_lag_top_k=1,

    use_cross_validation_if_small=False,min_validation_rows=3,min_test_rows=3,cv_folds=5,

    enable_placebo_tests=False,max_placebos=10,enable_jackknife_tests=False,max_jackknife_donors=10,

    base_random_state=42,base_save_dir="runtime_PC",
    save_results=True,plot_results=False,compact_summary=False,keep_all_results_in_memory=False,skip_existing_runs=True,
)

Running 12 experiment-grid runs with up to 24 selected model rows. Main save folder: runtime_PC
1/12 | features=standard_donors | controls=2 | rep=0

2/12 | features=treated_lags | controls=2 | rep=0
3/12 | features=treated_and_donor_lags | controls=2 | rep=0
4/12 | features=standard_donors | controls=50 | rep=0
5/12 | features=treated_lags | controls=50 | rep=0
6/12 | features=treated_and_donor_lags | controls=50 | rep=0
7/12 | features=standard_donors | controls=100 | rep=0
8/12 | features=treated_lags | controls=100 | rep=0
9/12 | features=treated_and_donor_lags | controls=100 | rep=0
10/12 | features=standard_donors | controls=all | rep=0
11/12 | features=treated_lags | controls=all | rep=0
12/12 | features=treated_and_donor_lags | controls=all | rep=0
Finished. Aggregated results saved in: runtime_PC
